# Almacén Robótico Autónomo - TE3002B

Simulación completa de un sistema multi-robot que coordina:
- **Husky A200** despeja obstáculos del corredor
- **ANYmal** transporta 3 PuzzleBots con marcha cuadrúpeda
- **PuzzleBots** apilan cajas cooperativamente con control de fuerza

---


## Detección de Entorno (Colab vs Local)

Detecta automáticamente si el código se ejecuta en Google Colab o localmente para configurar matplotlib correctamente.

In [10]:
# ============================================================================
# CONFIGURACIÓN AUTOMÁTICA DE ENTORNO
# ============================================================================

def detect_environment():
    """Detecta automáticamente si estamos en Colab o Local."""
    try:
        import google.colab
        return 'colab'
    except ImportError:
        return 'local'

# Detectar entorno
ENV = detect_environment()

# Configurar matplotlib según entorno
import matplotlib

if ENV == 'colab':
    matplotlib.use('Agg')  # Backend sin GUI para Colab
    from IPython.display import clear_output, display
    print("✓ Configurado para Google Colab")
    print("  Backend: Agg (sin GUI)")
else:
    try:
        matplotlib.use('TkAgg')  # Backend interactivo para local
        print("✓ Configurado para ejecución local")
        print("  Backend: TkAgg (interactivo)")
    except Exception as e:
        matplotlib.use('Agg')
        print(f"⚠ No se pudo usar TkAgg: {e}")
        print("  Backend: Agg (sin GUI)")

print(f"\nEntorno detectado: {ENV}")
print(f"Backend matplotlib: {matplotlib.get_backend()}")

✓ Configurado para ejecución local
  Backend: TkAgg (interactivo)

Entorno detectado: local
Backend matplotlib: TkAgg


## Librerías necesarias


In [11]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle, FancyBboxPatch, FancyArrowPatch
import time, os, json
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass, field
from enum import Enum, auto

print("Imports listos")

Imports listos


## Sistema de Logging de Torques

Registra y analiza los torques generados durante el control de fuerza.
Calcula estadísticas y detecta singularidades (det(J) < 1e-3).


In [13]:
"""
torque_logger.py - Utility for logging and analyzing torque data from force control

This module provides functionality to log torque data from the PuzzleBotArm
force control implementation and generate reports for the rubric requirements.
"""

import numpy as np
from typing import List, Dict
import json

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except ImportError:
    MATPLOTLIB_AVAILABLE = False
    print("[WARNING] matplotlib not available. Plotting disabled.")

class TorqueLogger:
    """Logger and analyzer for torque data from force control operations."""
    
    def __init__(self):
        self.torque_history: List[Dict] = []
        self.force_control_events: List[Dict] = []
        
    def log_torque_data(self, robot_id: str, operation: str, torques: np.ndarray, 
                       force_applied: np.ndarray, det_J: float, timestamp: float):
        """Log torque data from force control operation."""
        entry = {
            "robot_id": robot_id,
            "operation": operation,
            "timestamp": timestamp,
            "torques": torques.tolist(),
            "torque_magnitude": float(np.linalg.norm(torques)),
            "force_applied": force_applied.tolist(),
            "det_J": det_J,
            "max_torque": float(np.max(np.abs(torques))),
            "min_torque": float(np.min(np.abs(torques)))
        }
        self.torque_history.append(entry)
        
    def log_force_control_event(self, robot_id: str, event_type: str, 
                               box_name: str, details: Dict):
        """Log force control events for tracking."""
        event = {
            "robot_id": robot_id,
            "event_type": event_type,
            "box_name": box_name,
            "timestamp": len(self.force_control_events),
            "details": details
        }
        self.force_control_events.append(event)
        
    def generate_torque_report(self, output_path: str = "torque_report.json"):
        """Generate comprehensive torque report for rubric requirements."""
        report = {
            "summary": {
                "total_operations": len(self.torque_history),
                "total_events": len(self.force_control_events),
                "robots_involved": list(set(entry["robot_id"] for entry in self.torque_history))
            },
            "torque_statistics": self._calculate_torque_stats(),
            "force_control_events": self.force_control_events,
            "detailed_torque_log": self.torque_history
        }
        
        with open(output_path, 'w') as f:
            json.dump(report, f, indent=2)
            
        print(f"[TorqueLogger] Report saved to {output_path}")
        return report
        
    def _calculate_torque_stats(self) -> Dict:
        """Calculate statistical measures of torque data."""
        if not self.torque_history:
            return {}
            
        all_torques = np.array([entry["torques"] for entry in self.torque_history])
        torque_magnitudes = np.array([entry["torque_magnitude"] for entry in self.torque_history])
        
        stats = {
            "mean_torque_magnitude": float(np.mean(torque_magnitudes)),
            "max_torque_magnitude": float(np.max(torque_magnitudes)),
            "min_torque_magnitude": float(np.min(torque_magnitudes)),
            "std_torque_magnitude": float(np.std(torque_magnitudes)),
            "joint_torque_means": all_torques.mean(axis=0).tolist(),
            "joint_torque_stds": all_torques.std(axis=0).tolist(),
            "singularities_detected": sum(1 for entry in self.torque_history if entry["det_J"] < 1e-3)
        }
        
        return stats
        
    def plot_torque_analysis(self, output_path: str = "torque_analysis.png"):
        """Generate torque analysis plots for technical report."""
        if not MATPLOTLIB_AVAILABLE:
            print("[TorqueLogger] matplotlib not available. Skipping plot generation.")
            return
            
        if not self.torque_history:
            print("[TorqueLogger] No torque data to plot")
            return
            
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle("Análisis de Control de Fuerza - Torques (TE3002B)", fontsize=14, fontweight='bold')
        
        # Extract data
        timestamps = [entry["timestamp"] for entry in self.torque_history]
        torque_magnitudes = [entry["torque_magnitude"] for entry in self.torque_history]
        torques_array = np.array([entry["torques"] for entry in self.torque_history])
        det_J_values = [entry["det_J"] for entry in self.torque_history]
        
        # 1. Torque magnitude over time
        axes[0, 0].plot(timestamps, torque_magnitudes, 'b-', linewidth=2)
        axes[0, 0].set_title("Magnitud de Torque vs Tiempo")
        axes[0, 0].set_xlabel("Operación")
        axes[0, 0].set_ylabel("| torque | [N·m]")
        axes[0, 0].grid(True, alpha=0.3)
        
        # 2. Joint torques
        for i in range(3):
            axes[0, 1].plot(timestamps, torques_array[:, i], label=f'Joint {i+1}', linewidth=2)
        axes[0, 1].set_title("Torques por Articulación")
        axes[0, 1].set_xlabel("Operación")
        axes[0, 1].set_ylabel("Torque [N·m]")
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # 3. Determinant of Jacobian
        axes[1, 0].semilogy(timestamps, det_J_values, 'r-', linewidth=2)
        axes[1, 0].axhline(y=1e-3, color='red', linestyle='--', alpha=0.7, label='det(J)_min')
        axes[1, 0].set_title("Determinante del Jacobiano")
        axes[1, 0].set_xlabel("Operación")
        axes[1, 0].set_ylabel("det(J)")
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # 4. Torque distribution
        torque_means = torques_array.mean(axis=0)
        torque_stds = torques_array.std(axis=0)
        joint_labels = ['q1 (Base)', 'q2 (Hombro)', 'q3 (Codo)']
        
        x_pos = np.arange(len(joint_labels))
        axes[1, 1].bar(x_pos, torque_means, yerr=torque_stds, capsize=5, alpha=0.7)
        axes[1, 1].set_title("Distribución de Torques por Articulación")
        axes[1, 1].set_xlabel("Articulación")
        axes[1, 1].set_ylabel("Torque Promedio [N·m]")
        axes[1, 1].set_xticks(x_pos)
        axes[1, 1].set_xticklabels(joint_labels)
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"[TorqueLogger] Torque analysis plot saved to {output_path}")
        
    def print_summary(self):
        """Print summary of torque data for console output."""
        if not self.torque_history:
            print("[TorqueLogger] No torque data available")
            return
            
        stats = self._calculate_torque_stats()
        
        print("\n" + "="*60)
        print("  REPORTE DE CONTROL DE FUERZA - TORQUES")
        print("="*60)
        print(f"Operaciones totales: {len(self.torque_history)}")
        print(f"Magnitud promedio de torque: {stats['mean_torque_magnitude']:.4f} N·m")
        print(f"Magnitud máxima de torque: {stats['max_torque_magnitude']:.4f} N·m")
        print(f"Singularidades detectadas: {stats['singularities_detected']}")
        print(f"Torques promedio por articulación: {np.round(stats['joint_torque_means'], 4)} N·m")
        print("="*60)

# Global instance for use across modules
torque_logger = TorqueLogger()

## Modelos de Machine Learning

Cada robot usa un modelo ML simple:
- **Husky**: Logistic Regression para clasificar maniobras seguras
- **ANYmal**: Linear Regression para predecir tiempo de llegada
- **PuzzleBot**: K-Means para identificar zonas de trabajo
- **Coordinator**: Ridge Regression para estimar tiempo total de misión


In [14]:
"""
robot_ml.py — Modelos ML simples para los 4 robots.

Integración minimalista con sim.py:
1. Husky - Logistic Regression (clasificar seguridad)
2. ANYmal - Linear Regression (predecir tiempo)
3. PuzzleBot - K-Means (zonas de trabajo)
4. Coordinator - Ridge Regression (tiempo total)
"""

import numpy as np


# ===========================================================================
# 1. HUSKY - Logistic Regression (Safety Classifier)
# ===========================================================================

class HuskyML:
    """Clasificador simple de seguridad para Husky."""
    
    def __init__(self):
        self.weights = np.array([0.5, -0.3, 0.2])  # [min_range, velocity, angle]
        self.bias = 0.1
        
    def is_safe(self, min_lidar_range: float, velocity: float, angle_to_box: float) -> tuple:
        """Retorna (is_safe, confidence)."""
        features = np.array([min_lidar_range, velocity, abs(angle_to_box)])
        z = np.dot(self.weights, features) + self.bias
        prob = 1 / (1 + np.exp(-z))
        return prob > 0.5, prob


# ===========================================================================
# 2. ANYMAL - Linear Regression (Time Predictor)
# ===========================================================================

class ANYmalML:
    """Predictor simple de tiempo para ANYmal."""
    
    def __init__(self):
        # Modelo: time = a*distance + b*payload + c
        self.weights = np.array([3.0, 0.5])  # [distance, payload]
        self.bias = 5.0
        
    def predict_time(self, distance: float, payload_kg: float) -> float:
        """Predice tiempo en segundos."""
        features = np.array([distance, payload_kg])
        time_pred = np.dot(self.weights, features) + self.bias
        return max(time_pred, 1.0)


# ===========================================================================
# 3. PUZZLEBOT - K-Means (Zone Discovery)
# ===========================================================================

class PuzzleBotML:
    """Descubridor simple de zonas para PuzzleBot."""
    
    def __init__(self):
        # Zonas predefinidas (centros)
        self.zones = {
            "pickup": np.array([9.8, 3.2]),
            "stack": np.array([10.5, 3.6]),
            "wait": np.array([9.0, 3.6])
        }
        
    def identify_zone(self, position: np.ndarray) -> str:
        """Identifica la zona más cercana."""
        min_dist = float('inf')
        closest_zone = "unknown"
        
        for zone_name, zone_center in self.zones.items():
            dist = np.linalg.norm(position[:2] - zone_center)
            if dist < min_dist:
                min_dist = dist
                closest_zone = zone_name
                
        return closest_zone


# ===========================================================================
# 4. COORDINATOR - Ridge Regression (Mission Time Predictor)
# ===========================================================================

class CoordinatorML:
    """Predictor simple de tiempo total de misión."""
    
    def __init__(self):
        # Modelo: total_time = w1*phase1 + w2*phase2 + w3*phase3 + bias
        self.weights = np.array([1.1, 1.05, 1.15])  # Factores por fase
        self.bias = 10.0  # Overhead
        
    def predict_total_time(self, phase1_time: float, phase2_time: float, phase3_time: float) -> float:
        """Predice tiempo total."""
        features = np.array([phase1_time, phase2_time, phase3_time])
        total = np.dot(self.weights, features) + self.bias
        return total


# ===========================================================================
# Wrapper Global para sim.py
# ===========================================================================

class RobotMLSystem:
    """Sistema ML completo para todos los robots."""
    
    def __init__(self):
        self.husky_ml = HuskyML()
        self.anymal_ml = ANYmalML()
        self.puzzlebot_ml = PuzzleBotML()
        self.coordinator_ml = CoordinatorML()
        
        print("[ML] Sistema ML inicializado para 4 robots")
        
    # Métodos de acceso directo
    
    def husky_check_safety(self, min_range: float, velocity: float, angle: float):
        """Husky: Verificar si maniobra es segura."""
        return self.husky_ml.is_safe(min_range, velocity, angle)
    
    def anymal_predict_eta(self, distance: float, payload: float):
        """ANYmal: Predecir tiempo de llegada."""
        return self.anymal_ml.predict_time(distance, payload)
    
    def puzzlebot_get_zone(self, position: np.ndarray):
        """PuzzleBot: Identificar zona actual."""
        return self.puzzlebot_ml.identify_zone(position)
    
    def coordinator_predict_mission(self, t1: float, t2: float, t3: float):
        """Coordinator: Predecir tiempo total de misión."""
        return self.coordinator_ml.predict_total_time(t1, t2, t3)


# Instancia global para sim.py
ml_system = RobotMLSystem()

[ML] Sistema ML inicializado para 4 robots


## ANYmal - Robot Cuadrúpedo

Implementa marcha trote dinámica con:
- **FK/IK** por pata (3 DoF cada una)
- Detección de singularidades mediante det(J)
- Transporte de payload (3 PuzzleBots, aproximadamente 6 kg)


In [15]:
"""
anymal_gait.py — Generador de marcha trote del ANYmal con FK/IK por pata.

Monitorea det(J) para evitar singularidades.
Payload: 3 PuzzleBots ≈ 6 kg.
"""

import numpy as np
from typing import List, Tuple, Optional
from dataclasses import dataclass, field


# ---------------------------------------------------------------------------
# Parámetros del ANYmal
# ---------------------------------------------------------------------------
ANYMAL_BODY_LENGTH = 0.55   # m
ANYMAL_BODY_WIDTH  = 0.34   # m
ANYMAL_LEG_OFFSETS = {
    "LF": np.array([ ANYMAL_BODY_LENGTH / 2,  ANYMAL_BODY_WIDTH / 2, 0.0]),
    "RF": np.array([ ANYMAL_BODY_LENGTH / 2, -ANYMAL_BODY_WIDTH / 2, 0.0]),
    "LH": np.array([-ANYMAL_BODY_LENGTH / 2,  ANYMAL_BODY_WIDTH / 2, 0.0]),
    "RH": np.array([-ANYMAL_BODY_LENGTH / 2, -ANYMAL_BODY_WIDTH / 2, 0.0]),
}

L_HAA = 0.0   # offset lateral (simplificado)
L_HFE = 0.20  # fémur
L_KFE = 0.21  # tibia

H_NOMINAL = 0.42   # Altura nominal

# Singularidad: |det(J)| mínimo permitido
DET_J_MIN = 1e-3


@dataclass
class LegState:
    name: str
    q: np.ndarray = field(default_factory=lambda: np.array([0.0, 0.6, -1.4]))
    foot_pos_world: np.ndarray = field(default_factory=lambda: np.zeros(3))
    in_contact: bool = True
    det_J: float = 1.0


@dataclass
class ANYmalState:
    x: float = 0.0
    y: float = 0.0
    z: float = H_NOMINAL
    theta: float = 0.0
    legs: dict = field(default_factory=dict)
    payload_kg: float = 6.0

    @property
    def pos2d(self) -> np.ndarray:
        return np.array([self.x, self.y])

    @property
    def pose3d(self) -> np.ndarray:
        return np.array([self.x, self.y, self.z])


class LegKinematics:
    def __init__(self, leg_name: str, body_offset: np.ndarray):
        self.name = leg_name
        self.body_offset = body_offset
        self.l_hfe = L_HFE
        self.l_kfe = L_KFE
        self.sign_lat = 1.0 if "L" in leg_name else -1.0

    def forward_kinematics(self, q: np.ndarray, body_pos: np.ndarray, body_yaw: float) -> np.ndarray:
        q_haa, q_hfe, q_kfe = q

        # Posición local
        x_leg = self.l_hfe * np.sin(q_hfe) + self.l_kfe * np.sin(q_hfe + q_kfe)
        y_leg = self.sign_lat * L_HAA 
        z_leg = -(self.l_hfe * np.cos(q_hfe) + self.l_kfe * np.cos(q_hfe + q_kfe))

        foot_hip = np.array([x_leg, y_leg, z_leg])

        R = np.array([
            [np.cos(body_yaw), -np.sin(body_yaw), 0],
            [np.sin(body_yaw),  np.cos(body_yaw), 0],
            [0, 0, 1]
        ])

        return body_pos + R @ (self.body_offset + foot_hip)

    def inverse_kinematics(self, foot_world: np.ndarray, body_pos: np.ndarray, body_yaw: float) -> Optional[np.ndarray]:
        R = np.array([
            [np.cos(body_yaw), -np.sin(body_yaw), 0],
            [np.sin(body_yaw),  np.cos(body_yaw), 0],
            [0, 0, 1]
        ])
        foot_hip = R.T @ (foot_world - body_pos) - self.body_offset

        x, y, z = foot_hip

        q_haa = np.arctan2(y * self.sign_lat, -z + 1e-9)

        r = np.sqrt(x**2 + z**2)

        # Límite matemático para evitar singularidad (|det(J)| > 1e-3)
        cos_kfe = (r**2 - self.l_hfe**2 - self.l_kfe**2) / (2 * self.l_hfe * self.l_kfe)
        # Limitamos a 0.995 en vez de 1.0 para forzar que la rodilla nunca se estire a 0 rad
        cos_kfe = np.clip(cos_kfe, -1.0, 0.995) 
        
        q_kfe = -np.arccos(cos_kfe)  # Rodilla siempre flexionada

        beta = np.arctan2(-z, x)
        gamma = np.arctan2(self.l_kfe * np.sin(-q_kfe), self.l_hfe + self.l_kfe * np.cos(-q_kfe))
        q_hfe = beta - gamma

        return np.array([q_haa, q_hfe, q_kfe])

    def jacobian(self, q: np.ndarray) -> np.ndarray:
        # Jacobiano numérico consistente con la FK
        body_ref = np.zeros(3)
        yaw_ref = 0.0
        eps = 1e-5
        J = np.zeros((3, 3))
        for i in range(3):
            qp = q.copy(); qp[i] += eps
            qm = q.copy(); qm[i] -= eps
            fp = self.forward_kinematics(qp, body_ref, yaw_ref)
            fm = self.forward_kinematics(qm, body_ref, yaw_ref)
            J[:, i] = (fp - fm) / (2 * eps)
        return J

    def jacobian_det(self, q: np.ndarray) -> float:
        return abs(np.linalg.det(self.jacobian(q)))


class TrotGait:
    """Generador de marcha trote dinámico relativo al cuerpo."""

    def __init__(self, step_height: float = 0.08, step_duration: float = 0.5, dt: float = 0.01):
        self.step_height = step_height
        self.T = step_duration
        self.dt = dt
        self.phase = 0.0 

        self.diagonal_pairs = [("LF", "RH"), ("RF", "LH")]

    def advance(self, v_body: float) -> dict:
        """Calcula los offsets de los pies en el marco local del cuerpo."""
        self.phase = (self.phase + self.dt / self.T) % 1.0
        
        # El paso longitudinal depende de la velocidad del robot y el tiempo de apoyo (T/2)
        stride = v_body * (self.T / 2.0)
        offsets = {}

        for i, (leg_a, leg_b) in enumerate(self.diagonal_pairs):
            phi_leg = (self.phase + i * 0.5) % 1.0

            if phi_leg < 0.5:
                # Fase de vuelo (Swing): Mover pie hacia adelante
                t = phi_leg * 2.0 
                x = -stride / 2.0 + stride * t
                z = self.step_height * np.sin(np.pi * t)
            else:
                # Fase de apoyo (Stance): Mover pie hacia atrás relativo al cuerpo
                t = (phi_leg - 0.5) * 2.0
                x = stride / 2.0 - stride * t
                z = 0.0

            offset = np.array([x, 0.0, z])
            offsets[leg_a] = offset
            offsets[leg_b] = offset

        return offsets


class ANYmalGait:
    DEST = np.array([11.5, 4.8])  # Zona de trabajo

    def __init__(self, dt: float = 0.01):
        self.state = ANYmalState()
        self.dt = dt
        self.gait = TrotGait(dt=dt)

        self.leg_kin = {
            name: LegKinematics(name, offset)
            for name, offset in ANYMAL_LEG_OFFSETS.items()
        }

        self._init_feet()

        self.det_J_log: List[dict] = []
        self.pos_log: List[np.ndarray] = []
        self.singularity_events: List[dict] = []

    def _init_feet(self):
        for name, kin in self.leg_kin.items():
            q_init = np.array([0.0, 0.6, -1.4])
            foot = kin.forward_kinematics(q_init, self.state.pose3d, self.state.theta)
            self.state.legs[name] = LegState(name=name, q=q_init, foot_pos_world=foot)

    def _update_jacobians(self) -> bool:
        safe = True
        dets = {}
        for name, leg in self.state.legs.items():
            kin = self.leg_kin[name]
            d = kin.jacobian_det(leg.q)
            leg.det_J = d
            dets[name] = d
            if d < DET_J_MIN:
                safe = False
                event = {"t": len(self.det_J_log) * self.dt, "leg": name, "det_J": d}
                self.singularity_events.append(event)
        
        self.det_J_log.append(dets)
        return safe

    def _step(self, v: float, direction: np.ndarray):
        # 1. Integrar cuerpo
        self.state.x += v * direction[0] * self.dt
        self.state.y += v * direction[1] * self.dt
        self.state.theta = np.arctan2(direction[1], direction[0])

        R_body = np.array([
            [np.cos(self.state.theta), -np.sin(self.state.theta), 0],
            [np.sin(self.state.theta),  np.cos(self.state.theta), 0],
            [0, 0, 1]
        ])

        # 2. Obtener offsets locales (relativos al avance)
        offsets_local = self.gait.advance(v)

        # 3. Aplicar IK
        for name, leg in self.state.legs.items():
            kin = self.leg_kin[name]
            off_local = offsets_local[name]
            
            # Convertir el offset local a coordenadas del mundo sumándolo al offset de cadera
            foot_des_world = self.state.pose3d + R_body @ (ANYMAL_LEG_OFFSETS[name] + off_local)

            q = kin.inverse_kinematics(foot_des_world, self.state.pose3d, self.state.theta)
            if q is not None:
                leg.q = q
                leg.foot_pos_world = kin.forward_kinematics(q, self.state.pose3d, self.state.theta)

        self._update_jacobians()
        self.pos_log.append(self.state.pos2d.copy())

    def walk_to(self, x_goal: float, y_goal: float, v: float = 0.4, tol: float = 0.15) -> bool:
        max_steps = int(60 / self.dt)
        for step_i in range(max_steps):
            dx = x_goal - self.state.x
            dy = y_goal - self.state.y
            dist = np.hypot(dx, dy)

            if dist < tol:
                print(f"[ANYmal] ✓ Llegué a destino. Error={dist:.4f} m < tol={tol} m")
                return True

            direction = np.array([dx, dy]) / (dist + 1e-9)
            v_current = min(v, dist * 2.0)
            self._step(v_current, direction)

            if step_i % 500 == 0:
                print(f"[ANYmal] t={step_i*self.dt:.1f}s | pos=({self.state.x:.2f},{self.state.y:.2f}) | dist={dist:.2f}m")

        dist_final = np.hypot(x_goal - self.state.x, y_goal - self.state.y)
        print(f"[ANYmal] ✗ Tiempo agotado. Error final={dist_final:.4f} m")
        return dist_final < tol

    def transport_puzzlebots(self) -> bool:
        print("\n" + "=" * 50)
        print("  FASE 2: Transporte ANYmal (trote)")
        print(f"  Payload: {self.state.payload_kg} kg  |  Destino: {self.DEST}")
        print("=" * 50)

        success = self.walk_to(self.DEST[0], self.DEST[1])

        n_sing = len(self.singularity_events)
        if n_sing == 0:
            print("[ANYmal] ✓ Sin singularidades durante el recorrido.")
        else:
            print(f"[ANYmal] ⚠ {n_sing} eventos de singularidad detectados:")
            for ev in self.singularity_events[:5]:
                print(f"   t={ev['t']:.2f}s | pata={ev['leg']} | det(J)={ev['det_J']:.2e}")

        print(f"[ANYmal] Pose final: ({self.state.x:.3f}, {self.state.y:.3f})")
        return success

    def det_J_summary(self) -> dict:
        if not self.det_J_log:
            return {}
        summary = {}
        for name in self.state.legs:
            vals = [d[name] for d in self.det_J_log]
            summary[name] = {
                "min": float(np.min(vals)),
                "mean": float(np.mean(vals)),
                "violations": int(np.sum(np.array(vals) < DET_J_MIN)),
            }
        return summary

## Husky A200 - Robot Móvil

Robot diferencial con:
- Control skid-steer con compensación de deslizamiento
- LiDAR 2D simulado para detección de obstáculos
- Máquina de estados para empujar cajas


In [16]:
"""
husky_pusher.py — Nodo que comanda el Husky A200 para empujar cajas.

Modelo: skid-steer con compensación de deslizamiento.
Sensor: LiDAR 2D simulado.
"""

import numpy as np
import time
from typing import List, Tuple, Dict


# ---------------------------------------------------------------------------
# Constantes del Husky A200
# ---------------------------------------------------------------------------
HUSKY_WIDTH = 0.67      # m, separación entre centros de ruedas
HUSKY_MAX_V = 1.0       # m/s
HUSKY_MAX_W = 1.5       # rad/s


class HuskyState:
    """Estado cinemático del Husky."""

    def __init__(self, x: float = 0.0, y: float = 0.0, theta: float = 0.0):
        self.x = x
        self.y = y
        self.theta = theta
        self.v_cmd = 0.0
        self.w_cmd = 0.0
        self.v_meas = 0.0
        self.w_meas = 0.0

    @property
    def pose(self) -> np.ndarray:
        return np.array([self.x, self.y, self.theta])

    def __repr__(self):
        return (f"Husky(x={self.x:.3f}, y={self.y:.3f}, θ={np.degrees(self.theta):.1f}°, "
                f"v={self.v_cmd:.3f}, ω={self.w_cmd:.3f})")


class Box:
    """Caja grande (obstáculo) en el corredor."""

    def __init__(self, box_id: str, x: float, y: float, width: float = 0.4, height: float = 0.4):
        self.id = box_id
        self.x = x
        self.y = y
        self.width = width
        self.height = height
        self.cleared = False

    @property
    def pos(self) -> np.ndarray:
        return np.array([self.x, self.y])


class LiDAR2D:
    """LiDAR 2D simulado para detección de cajas."""

    def __init__(self, range_max: float = 8.0, n_beams: int = 180, noise_std: float = 0.01):
        self.range_max = range_max
        self.n_beams = n_beams
        self.noise_std = noise_std
        self.angles = np.linspace(-np.pi / 2, np.pi / 2, n_beams)

    def scan(self, robot_state: HuskyState, boxes: List[Box]) -> np.ndarray:
        ranges = np.full(self.n_beams, self.range_max)

        for i, angle_local in enumerate(self.angles):
            angle_world = robot_state.theta + angle_local
            dx = np.cos(angle_world)
            dy = np.sin(angle_world)

            for box in boxes:
                if box.cleared:
                    continue
                d = self._ray_aabb(
                    robot_state.x, robot_state.y, dx, dy,
                    box.x - box.width / 2, box.y - box.height / 2,
                    box.x + box.width / 2, box.y + box.height / 2,
                )
                if d is not None and d < ranges[i]:
                    ranges[i] = d

        ranges += np.random.normal(0, self.noise_std, self.n_beams)
        ranges = np.clip(ranges, 0, self.range_max)
        return ranges

    @staticmethod
    def _ray_aabb(
        ox: float, oy: float, dx: float, dy: float,
        x_min: float, y_min: float, x_max: float, y_max: float
    ) -> float | None:
        t_min, t_max = 0.0, 1e9
        for o, d, lo, hi in [(ox, dx, x_min, x_max), (oy, dy, y_min, y_max)]:
            if abs(d) < 1e-10:
                if o < lo or o > hi:
                    return None
            else:
                t1 = (lo - o) / d
                t2 = (hi - o) / d
                t_min = max(t_min, min(t1, t2))
                t_max = min(t_max, max(t1, t2))
        if t_min > t_max or t_max < 0:
            return None
        return t_min if t_min >= 0 else t_max


class SkidSteerController:
    """Controlador skid-steer con compensación de deslizamiento."""

    def __init__(self, slip_factor: float = 0.05):
        self.slip = slip_factor
        self.kp_v = 1.0
        self.kp_w = 2.0

        self.log_v_cmd: List[float] = []
        self.log_w_cmd: List[float] = []
        self.log_v_meas: List[float] = []
        self.log_w_meas: List[float] = []

    def compute(
        self, state: HuskyState, v_des: float, w_des: float, dt: float
    ) -> Tuple[float, float]:
        v_cmd = np.clip(v_des, -HUSKY_MAX_V, HUSKY_MAX_V)
        w_cmd = np.clip(w_des, -HUSKY_MAX_W, HUSKY_MAX_W)

        v_meas = v_cmd * (1 - self.slip) + np.random.normal(0, 0.005)
        w_meas = w_cmd * (1 - self.slip * 0.5) + np.random.normal(0, 0.005)

        v_comp = v_cmd / (1 - self.slip + 1e-9)
        w_comp = w_cmd / (1 - self.slip * 0.5 + 1e-9)
        v_comp = np.clip(v_comp, -HUSKY_MAX_V, HUSKY_MAX_V)
        w_comp = np.clip(w_comp, -HUSKY_MAX_W, HUSKY_MAX_W)

        state.v_cmd = v_comp
        state.w_cmd = w_comp
        state.v_meas = v_meas
        state.w_meas = w_meas

        self.log_v_cmd.append(v_comp)
        self.log_w_cmd.append(w_comp)
        self.log_v_meas.append(v_meas)
        self.log_w_meas.append(w_meas)

        return v_comp, w_comp

    def report(self):
        if not self.log_v_cmd:
            return
        v_err = np.abs(np.array(self.log_v_cmd) - np.array(self.log_v_meas))
        w_err = np.abs(np.array(self.log_w_cmd) - np.array(self.log_w_meas))
        print(f"[SkidSteer] Error v: μ={v_err.mean():.4f}, σ={v_err.std():.4f} m/s")
        print(f"[SkidSteer] Error ω: μ={w_err.mean():.4f}, σ={w_err.std():.4f} rad/s")


class HuskyPusher:
    """Nodo principal del Husky A200 para empujar cajas del corredor."""

    CORRIDOR_X_MIN = 1.0
    CORRIDOR_X_MAX = 7.0
    CORRIDOR_Y_MIN = -1.0
    CORRIDOR_Y_MAX = 1.0

    def __init__(self, slip_factor: float = 0.05, dt: float = 0.05):
        self.state = HuskyState(x=0.0, y=0.0, theta=0.0)
        self.controller = SkidSteerController(slip_factor)
        self.lidar = LiDAR2D()
        self.dt = dt

        self.boxes = [
            Box("B1", x=2.5, y=0.0),
            Box("B2", x=4.0, y=0.3),
            Box("B3", x=5.5, y=-0.2),
        ]

        self.phase_log: List[str] = []
        self.time = 0.0
        
        self.nav_state = "IDLE"
        self.target_box = None
        self.target_pos = None
        self.pushing = False

    def detect_boxes(self) -> List[Dict]:
        ranges = self.lidar.scan(self.state, self.boxes)
        detected = []
        for i, r in enumerate(ranges):
            if r < self.lidar.range_max - 0.1:
                angle = self.state.theta + self.lidar.angles[i]
                bx = self.state.x + r * np.cos(angle)
                by = self.state.y + r * np.sin(angle)
                detected.append({"range": r, "angle": self.lidar.angles[i],
                                  "world_x": bx, "world_y": by})
        return detected

    def goto(
        self, x_goal: float, y_goal: float,
        tol_pos: float = 0.15, max_steps: int = 500,
        pushing: bool = False, current_box: Box = None
    ) -> bool:
        """Navega de forma bloqueante (con sincronización de tiempo real para animación)."""
        for _ in range(max_steps):
            dx = x_goal - self.state.x
            dy = y_goal - self.state.y
            dist = np.hypot(dx, dy)

            if dist < tol_pos:
                return True

            angle_to_goal = np.arctan2(dy, dx)
            angle_err = self._wrap_angle(angle_to_goal - self.state.theta)

            if abs(angle_err) < 0.5:
                v_des = min(0.8 * dist, HUSKY_MAX_V)
            else:
                v_des = 0.0

            w_des = 2.5 * angle_err

            v_cmd, w_cmd = self.controller.compute(self.state, v_des, w_des, self.dt)
            self._integrate(v_cmd, w_cmd)
            self.time += self.dt
            
            time.sleep(self.dt * 0.5) 

            # LÓGICA CORREGIDA: FÍSICA DE CONTACTO REAL
            if pushing and current_box is not None:
                contact_distance = 0.5 # Aumentado para mejor colisión visual
                dist_to_box = np.hypot(current_box.x - self.state.x, current_box.y - self.state.y)
                
                # Solo mueve la caja si el Husky ya llegó a tocarla físicamente
                if dist_to_box <= contact_distance + 0.05:
                    current_box.x = self.state.x + contact_distance * np.cos(self.state.theta)
                    current_box.y = self.state.y + contact_distance * np.sin(self.state.theta)

        return False

    def _integrate(self, v: float, w: float):
        self.state.theta += w * self.dt
        self.state.theta = self._wrap_angle(self.state.theta)
        self.state.x += v * np.cos(self.state.theta) * self.dt
        self.state.y += v * np.sin(self.state.theta) * self.dt

    @staticmethod
    def _wrap_angle(a: float) -> float:
        return (a + np.pi) % (2 * np.pi) - np.pi

    # ------------------------------------------------------------------
    # Lógica No Bloqueante
    # ------------------------------------------------------------------

    def push_box_nonblocking(self, box: Box) -> bool:
        """Versión no bloqueante que usa un PRE_POSITIONING para esquivar colisiones."""
        if self.nav_state == "IDLE":
            print(f"\n[HuskyPusher] Iniciando aproximación a caja {box.id}")
            self.target_box = box
            
            push_dir_y = 1.0 if box.y >= 0 else -1.0
            push_target_y = push_dir_y * (self.CORRIDOR_Y_MAX + box.height + 0.3)
            
            # Offset de preparación ampliado para dar espacio de aproximación
            offset = 0.7 
            
            self.target_pos = {
                "pre_behind": np.array([box.x - 0.8, box.y - (push_dir_y * offset)]),
                "behind": np.array([box.x, box.y - (push_dir_y * offset)]),
                "push": np.array([box.x, push_target_y])
            }
            
            self.nav_state = "PRE_POSITIONING"
            return False

        elif self.nav_state == "PRE_POSITIONING":
            target = self.target_pos["pre_behind"]
            if self._step_goto(target[0], target[1], tol_pos=0.15):
                print(f"[HuskyPusher] En punto seguro. Alineándose con {self.target_box.id}")
                self.nav_state = "POSITIONING"
            return False
            
        elif self.nav_state == "POSITIONING":
            target = self.target_pos["behind"]
            if self._step_goto(target[0], target[1], tol_pos=0.1):
                print(f"[HuskyPusher] Posicionado. Empujando...")
                self.nav_state = "PUSHING"
                self.pushing = True
            return False
            
        elif self.nav_state == "PUSHING":
            target = self.target_pos["push"]
            if self._step_goto(target[0], target[1], tol_pos=0.2, pushing=True):
                self.pushing = False
                
                out = (self.target_box.y < self.CORRIDOR_Y_MIN - self.target_box.height / 2
                       or self.target_box.y > self.CORRIDOR_Y_MAX + self.target_box.height / 2)
                self.target_box.cleared = out
                
                if out:
                    print(f"[HuskyPusher] ✓ Caja {self.target_box.id} despejada")
                else:
                    print(f"[HuskyPusher] ✗ Caja {self.target_box.id} sigue en corredor")
                
                self.nav_state = "IDLE"
                self.target_box = None
                self.target_pos = None
                return True
            return False
            
        return False
    
    def _step_goto(self, x_goal: float, y_goal: float, tol_pos: float = 0.15, 
                   pushing: bool = False) -> bool:
        """Paso único de navegación para animaciones no bloqueantes."""
        dx = x_goal - self.state.x
        dy = y_goal - self.state.y
        dist = np.hypot(dx, dy)

        if dist < tol_pos:
            return True

        angle_to_goal = np.arctan2(dy, dx)
        angle_err = self._wrap_angle(angle_to_goal - self.state.theta)

        if abs(angle_err) < 0.5:
            v_des = min(0.8 * dist, HUSKY_MAX_V)
        else:
            v_des = 0.0

        w_des = 2.5 * angle_err

        v_cmd, w_cmd = self.controller.compute(self.state, v_des, w_des, self.dt)
        self._integrate(v_cmd, w_cmd)
        self.time += self.dt

        # LÓGICA CORREGIDA: FÍSICA DE CONTACTO REAL
        if pushing and self.target_box is not None:
            contact_distance = 0.5 # Aumentado para mejor colisión visual
            dist_to_box = np.hypot(self.target_box.x - self.state.x, self.target_box.y - self.state.y)
            
            # Solo mueve la caja si el Husky ya llegó a tocarla físicamente
            if dist_to_box <= contact_distance + 0.05:
                self.target_box.x = self.state.x + contact_distance * np.cos(self.state.theta)
                self.target_box.y = self.state.y + contact_distance * np.sin(self.state.theta)

        return False

    def clear_corridor_step(self) -> bool:
        """Iteración global de la máquina de estados."""
        if self.nav_state == "IDLE":
            for box in self.boxes:
                if not box.cleared:
                    self.push_box_nonblocking(box)
                    return False
            
            self.nav_state = "RETURNING_HOME"
            self.target_pos = {"home": np.array([0.5, 0.0])}
            return False
            
        elif self.nav_state == "RETURNING_HOME":
            target = self.target_pos["home"]
            if self._step_goto(target[0], target[1], tol_pos=0.15):
                self.nav_state = "DONE"
                corridor_clear = all(box.cleared for box in self.boxes)
                print(f"\n[HuskyPusher] Corredor {'✓ DESPEJADO' if corridor_clear else '✗ NO despejado'}")
                return True
            return False
            
        elif self.nav_state == "DONE":
            return True
            
        else:
            self.push_box_nonblocking(self.target_box)
            
        return False
        
    # ------------------------------------------------------------------
    # Ejecución Bloqueante (Usada por el Coordinator heredado)
    # ------------------------------------------------------------------
    
    def push_box(self, box: Box) -> bool:
        """Versión bloqueante con puntos intermedios."""
        print(f"\n[HuskyPusher] Aproximación segura a caja {box.id}")

        push_dir_y = 1.0 if box.y >= 0 else -1.0
        push_target_y = push_dir_y * (self.CORRIDOR_Y_MAX + box.height + 0.3)

        offset = 0.7 
        behind_y = box.y - (push_dir_y * offset)
        
        pre_behind_x = box.x - 0.8
        ok = self.goto(pre_behind_x, behind_y, tol_pos=0.15)
        if not ok: return False
            
        behind_x = box.x
        ok = self.goto(behind_x, behind_y, tol_pos=0.1)
        if not ok:
            print(f"[HuskyPusher] No pude posicionarme detrás de {box.id}")
            return False

        push_x = box.x
        push_y = push_target_y
        ok = self.goto(push_x, push_y, tol_pos=0.2, pushing=True, current_box=box)

        out = (box.y < self.CORRIDOR_Y_MIN - box.height / 2
               or box.y > self.CORRIDOR_Y_MAX + box.height / 2)
        box.cleared = out

        if out:
            print(f"[HuskyPusher] ✓ Caja {box.id} fuera del corredor → ({box.x:.2f}, {box.y:.2f})")
        else:
            print(f"[HuskyPusher] ✗ Caja {box.id} sigue en el corredor")

        return out

    def clear_corridor(self) -> bool:
        print("\n" + "=" * 50)
        print("  FASE 1: Despeje del corredor — Husky A200")
        print("=" * 50)

        all_clear = True
        for box in self.boxes:
            success = self.push_box(box)
            self.phase_log.append(
                f"t={self.time:.1f}s | Caja {box.id}: {'DESPEJADA' if success else 'FALLO'}"
            )
            if not success:
                all_clear = False

        self.goto(0.5, 0.0)

        print("\n[HuskyPusher] Reporte final:")
        for entry in self.phase_log:
            print(f"  {entry}")
        self.controller.report()

        corridor_clear = all(box.cleared for box in self.boxes)
        print(f"\n[HuskyPusher] Corredor {'✓ DESPEJADO' if corridor_clear else '✗ NO despejado'}")
        return corridor_clear

    def get_state(self) -> HuskyState:
        return self.state

    def get_boxes(self) -> List[Box]:
        return self.boxes

## PuzzleBot Arm - Brazo Manipulador

Mini brazo planar de 3 DoF:
- q1: rotación base (yaw)
- q2, q3: eslabones en plano vertical
- **Control de fuerza**: τ = J^T · f (Jacobiano transpuesto)


In [17]:
"""
puzzlebot_arm.py — Mini brazo planar de 3 DoF del PuzzleBot.

Configuración: base rotacional (q1) + 2 eslabones en plano vertical (q2, q3).
Unidades: metros, radianes, Newtons, N·m.
"""

import numpy as np
from typing import Tuple, Optional, List


class PuzzleBotArm:
    """Mini brazo planar de 3 DoF montado sobre un PuzzleBot.

    Configuración:
        - q1: rotación de la base (yaw, en plano XY)
        - q2: ángulo del primer eslabón (hombro, en plano vertical)
        - q3: ángulo del segundo eslabón (codo, en plano vertical)

    Marco de referencia: origen en la base del brazo (montado sobre el PuzzleBot).
    """

    def __init__(self, l1: float = 0.05, l2: float = 0.12, l3: float = 0.10):
        """
        Args:
            l1: Altura de la base al hombro [m].
            l2: Longitud del primer eslabón dinámico [m].
            l3: Longitud del segundo eslabón dinámico [m].
        """
        self.l1 = l1
        self.l2 = l2
        self.l3 = l3
        self.q = np.zeros(3)  # [q1, q2, q3] en radianes

        # Límites articulares [rad]
        self.q_min = np.array([-np.pi, -np.pi / 2, -np.pi * 0.9])
        self.q_max = np.array([np.pi, np.pi, np.pi * 0.9])

        # Estado de agarre
        self.grasping = False
        self.grip_force = 0.0

        # Historial para logging
        self.torque_log: List[np.ndarray] = []
        self.pose_log: List[np.ndarray] = []
        self.force_control_log: List[dict] = []  # Log for force control data
        self.time = 0.0  # Timestamp for logging

    # ------------------------------------------------------------------
    # Cinemática Directa (FK)
    # ------------------------------------------------------------------

    def forward_kinematics(self, q: Optional[np.ndarray] = None) -> np.ndarray:
        if q is not None:
            self.q = np.clip(q, self.q_min, self.q_max)

        q1, q2, q3 = self.q

        # Proyección horizontal del brazo (radio desde el eje de q1)
        r = self.l2 * np.cos(q2) + self.l3 * np.cos(q2 + q3)

        # Posición 3D
        x = r * np.cos(q1)
        y = r * np.sin(q1)
        z = self.l1 + self.l2 * np.sin(q2) + self.l3 * np.sin(q2 + q3)

        p = np.array([x, y, z])
        self.pose_log.append(p.copy())
        return p

    # ------------------------------------------------------------------
    # Cinemática Inversa (IK)
    # ------------------------------------------------------------------

    def inverse_kinematics(self, p_des: np.ndarray) -> Optional[np.ndarray]:
        x, y, z = p_des

        # --- q1: ángulo de la base (yaw) ---
        q1 = np.arctan2(y, x)

        # --- Resolución en el plano vertical ---
        r = np.sqrt(x**2 + y**2)
        z_prime = z - self.l1  # Restamos la altura de la base

        D_sq = r**2 + z_prime**2
        D = np.sqrt(D_sq)

        # Verificar alcanzabilidad
        if D > self.l2 + self.l3 + 1e-6:
            print(f"[PuzzleBotArm] IK: punto fuera del workspace (D={D:.4f} > {self.l2 + self.l3:.4f})")
            return None
        if D < abs(self.l2 - self.l3) - 1e-6:
            print(f"[PuzzleBotArm] IK: punto demasiado cercano (D={D:.4f})")
            return None

        # Ley de cosenos para q3
        cos_q3 = (D_sq - self.l2**2 - self.l3**2) / (2 * self.l2 * self.l3)
        cos_q3 = np.clip(cos_q3, -1.0, 1.0)
        q3 = -np.arccos(cos_q3)  # Codo hacia abajo (o arriba, según convención)

        # Geometría para q2
        beta = np.arctan2(z_prime, r)
        gamma = np.arctan2(self.l3 * np.sin(abs(q3)), self.l2 + self.l3 * np.cos(q3))
        
        if q3 < 0:
            q2 = beta + gamma
        else:
            q2 = beta - gamma

        q = np.array([q1, q2, q3])
        q = np.clip(q, self.q_min, self.q_max)
        self.q = q
        return q

    # ------------------------------------------------------------------
    # Jacobiano Analítico
    # ------------------------------------------------------------------

    def jacobian(self, q: Optional[np.ndarray] = None) -> np.ndarray:
        if q is not None:
            self.q = q

        q1, q2, q3 = self.q

        # Radio y sus derivadas
        r = self.l2 * np.cos(q2) + self.l3 * np.cos(q2 + q3)
        dr_dq2 = -self.l2 * np.sin(q2) - self.l3 * np.sin(q2 + q3)
        dr_dq3 = -self.l3 * np.sin(q2 + q3)

        # Derivadas de Z
        dz_dq2 = self.l2 * np.cos(q2) + self.l3 * np.cos(q2 + q3)
        dz_dq3 = self.l3 * np.cos(q2 + q3)

        J = np.array([
            [-r * np.sin(q1),   dr_dq2 * np.cos(q1),   dr_dq3 * np.cos(q1)],
            [ r * np.cos(q1),   dr_dq2 * np.sin(q1),   dr_dq3 * np.sin(q1)],
            [ 0.0,              dz_dq2,                dz_dq3             ],
        ])
        return J

    def jacobian_det(self, q: Optional[np.ndarray] = None) -> float:
        return abs(np.linalg.det(self.jacobian(q)))

    # ------------------------------------------------------------------
    # Control de Fuerza
    # ------------------------------------------------------------------

    def force_to_torque(self, f_tip: np.ndarray) -> np.ndarray:
        J = self.jacobian()
        tau = J.T @ f_tip
        self.torque_log.append(tau.copy())
        return tau

    # ------------------------------------------------------------------
    # Trayectoria Cartesiana + Agarre
    # ------------------------------------------------------------------

    def _cartesian_trajectory(self, p_start: np.ndarray, p_end: np.ndarray, steps: int = 20) -> List[np.ndarray]:
        return [p_start + t * (p_end - p_start) for t in np.linspace(0, 1, steps)]

    def grasp_box(self, box_pos: np.ndarray, grip_force: float = 5.0, steps: int = 30) -> bool:
        p_current = self.forward_kinematics()

        # Aproximación por encima de la caja
        p_pregrasp = box_pos + np.array([0, 0, 0.04])
        traj = self._cartesian_trajectory(p_current, p_pregrasp, steps)

        for p in traj:
            q = self.inverse_kinematics(p)
            if q is None:
                print(f"[PuzzleBotArm] grasp_box: IK falló en trayectoria pre-grasp.")
                return False

        # Descender al punto de agarre
        p_grasp = box_pos.copy()
        traj_down = self._cartesian_trajectory(p_pregrasp, p_grasp, 10)
        for p in traj_down:
            q = self.inverse_kinematics(p)
            if q is None:
                return False

        # Aplicar fuerza
        f_grip = np.array([0.0, 0.0, -grip_force])
        tau = self.force_to_torque(f_grip)

        det_J = self.jacobian_det()
        if det_J < 1e-3:
            print(f"[PuzzleBotArm] ¡ADVERTENCIA! Singularidad en agarre: det(J)={det_J:.6f}")

        self.grasping = True
        self.grip_force = grip_force
        print(f"[PuzzleBotArm] Agarre exitoso en {np.round(box_pos, 3)}. τ={np.round(tau, 4)} N·m | det(J)={det_J:.5f}")
        return True

    def place_box(self, target_pos: np.ndarray, steps: int = 30) -> bool:
        if not self.grasping:
            print("[PuzzleBotArm] place_box: no hay caja agarrada.")
            return False

        p_current = self.forward_kinematics()
        p_up = p_current + np.array([0, 0, 0.05])
        
        # Lift box
        for p in self._cartesian_trajectory(p_current, p_up, 10):
            if self.inverse_kinematics(p) is None: return False

        # Move above target
        p_above_target = target_pos + np.array([0, 0, 0.05])
        for p in self._cartesian_trajectory(p_up, p_above_target, steps):
            if self.inverse_kinematics(p) is None: return False

        # Descend with force control using Jacobian Transposed (τ = J^T * f)
        descent_steps = 10
        contact_force = np.array([0.0, 0.0, -self.grip_force * 0.5])  # Contact force
        
        for i, p in enumerate(self._cartesian_trajectory(p_above_target, target_pos, descent_steps)):
            if self.inverse_kinematics(p) is None: return False
            
            # Apply force control using Jacobian Transposed
            # τ = J^T * f where f is the desired contact force
            J = self.jacobian()
            tau_contact = J.T @ contact_force
            
            # Log the torques for rubric requirements
            self.torque_log.append(tau_contact.copy())
            
            # Check for singularity
            det_J = self.jacobian_det()
            if det_J < 1e-3:
                print(f"[PuzzleBotArm] ¡ADVERTENCIA! Singularidad en colocación: det(J)={det_J:.6f}")
            
            # Log force control information
            if i == descent_steps - 1:  # Final contact
                print(f"[PuzzleBotArm] Control de fuerza - Contacto suave:")
                print(f"  Fuerza aplicada: {contact_force} N")
                print(f"  Torques calculados (τ=J^T*f): {np.round(tau_contact, 4)} N·m")
                print(f"  det(J) = {det_J:.5f}")
                
                # Log to torque logger for rubric requirements
                torque_logger.log_torque_data(
                    robot_id=f"PuzzleBot_arm",
                    operation="place_box_contact",
                    torques=tau_contact,
                    force_applied=contact_force,
                    det_J=det_J,
                    timestamp=self.time
                )
                
                torque_logger.log_force_control_event(
                    robot_id=f"PuzzleBot_arm",
                    event_type="contact_force_control",
                    box_name="target",
                    details={
                        "method": "jacobian_transposed",
                        "formula": "τ = J^T * f",
                        "contact_force": contact_force.tolist(),
                        "resulting_torques": tau_contact.tolist()
                    }
                )

        self.grasping = False
        self.grip_force = 0.0
        print(f"[PuzzleBotArm] Caja colocada en {np.round(target_pos, 3)} con control de fuerza.")
        return True

    def reset(self):
        self.q = np.zeros(3)
        self.grasping = False
        self.grip_force = 0.0

## Coordinator - Orquestador

Máquina de estados que coordina las 3 fases:
1. Husky despeja corredor
2. ANYmal transporta PuzzleBots
3. PuzzleBots apilan cajas (C→B→A) con sincronización por eventos


In [18]:
"""
coordinator.py — Máquina de estados que orquesta las tres fases del reto.

Fases:
    IDLE → PHASE1_HUSKY → PHASE2_ANYMAL → XARM_UNLOAD → PHASE3_PUZZLEBOTS → DONE
"""

import numpy as np
import time
from enum import Enum, auto
from typing import List, Optional, Tuple
from dataclasses import dataclass

# Imports directos a los módulos en la misma carpeta


# ---------------------------------------------------------------------------
# Estado del sistema
# ---------------------------------------------------------------------------

class Phase(Enum):
    IDLE            = auto()
    PHASE1_HUSKY    = auto()
    PHASE2_ANYMAL   = auto()
    XARM_UNLOAD     = auto()   # Extra: XArm baja los PuzzleBots
    PHASE3_PUZZLEBOTS = auto()
    DONE            = auto()
    ERROR           = auto()


@dataclass
class SmallBox:
    """Caja pequeña que deben apilar los PuzzleBots."""
    name: str     # "A", "B", "C"
    pos: np.ndarray
    stacked: bool = False
    stack_height: float = 0.0  # altura en la pila [m]

    BOX_HEIGHT = 0.05  # m


# ---------------------------------------------------------------------------
# XArm — manipulador de descarga (Puntos extra)
# ---------------------------------------------------------------------------

class XArm:
    """XArm de 6 DoF (simplificado) para bajar los PuzzleBots del ANYmal.

    Modelo: cinemática de alcance adaptada, montado junto a la zona de trabajo.
    """

    # Dimensiones básicas (eslabones adaptados a la simulación)
    LINK_LENGTHS = [0.267, 0.289, 0.078, 0.343, 0.076, 0.097]  # m

    def __init__(self, arm_id: int, base_pos: np.ndarray):
        self.id = arm_id
        self.base_pos = base_pos
        self.q = np.zeros(6)       # Articulaciones [rad]
        self.payload: Optional[str] = None  # Nombre del PuzzleBot cargado

    def _ik_simple(self, target_world: np.ndarray) -> np.ndarray:
        """IK simplificada: calcula ángulos para alcanzar target_world."""
        delta = target_world - self.base_pos
        r = np.linalg.norm(delta[:2])
        z = delta[2]

        # Ángulo de la base
        q1 = np.arctan2(delta[1], delta[0])

        # Alcance en el plano (r, z) para eslabones 2 y 3 de la cadena
        L1 = self.LINK_LENGTHS[1]  # 0.289
        L2 = self.LINK_LENGTHS[3]  # 0.343
        
        # Distancia euclidiana directa al objetivo
        D = np.sqrt(r**2 + z**2)
        
        # Limitar matemáticamente D para evitar singularidad de frontera (NaN)
        D = np.clip(D, 0.01, L1 + L2 - 0.005)

        # Ley de cosenos protegida
        cos_q3 = (D**2 - L1**2 - L2**2) / (2 * L1 * L2)
        q3 = -np.arccos(np.clip(cos_q3, -1.0, 1.0))

        beta = np.arctan2(z, r)
        gamma = np.arctan2(L2 * np.sin(-q3), L1 + L2 * np.cos(-q3))
        q2 = beta + gamma

        q = np.array([q1, q2, 0.0, q3, 0.0, 0.0])
        self.q = q
        return q

    def pick_from_anymal(self, puzzlebot_id: int, anymal_pos: np.ndarray) -> bool:
        # PuzzleBots están espaciados 0.15 m en el dorso del ANYmal
        offset_x = (puzzlebot_id - 1) * 0.15
        pick_pos = np.array([
            anymal_pos[0] + offset_x,
            anymal_pos[1],
            0.42 + 0.12  # Altura del dorso del ANYmal + altura PuzzleBot
        ])

        dist = np.linalg.norm(pick_pos - self.base_pos)
        reach = self.LINK_LENGTHS[1] + self.LINK_LENGTHS[3]
        if dist > reach:
            print(f"[XArm{self.id}] PuzzleBot {puzzlebot_id} fuera de alcance (dist={dist:.3f} > {reach:.3f} m)")
            return False

        q = self._ik_simple(pick_pos)
        print(f"[XArm{self.id}] Recogiendo PuzzleBot {puzzlebot_id} desde ({pick_pos[0]:.2f}, {pick_pos[1]:.2f}, {pick_pos[2]:.2f})")
        print(f"[XArm{self.id}] q = {np.round(np.degrees(q[:4]), 1)}° (primeras 4 articulaciones)")
        self.payload = f"PuzzleBot_{puzzlebot_id}"
        return True

    def place_on_table(self, table_pos: np.ndarray) -> bool:
        if self.payload is None:
            print(f"[XArm{self.id}] No hay payload.")
            return False

        q = self._ik_simple(table_pos)
        print(f"[XArm{self.id}] Colocando {self.payload} en mesa ({table_pos[0]:.2f}, {table_pos[1]:.2f}, {table_pos[2]:.2f})")
        print(f"[XArm{self.id}] q = {np.round(np.degrees(q[:4]), 1)}°")
        self.payload = None
        return True


# ---------------------------------------------------------------------------
# PuzzleBot completo (robot móvil + brazo)
# ---------------------------------------------------------------------------

class PuzzleBot:
    """PuzzleBot: robot diferencial con mini brazo 3 DoF."""

    TABLE_BOXES_POS = {
        "A": np.array([9.5, 3.2, 0.02]),
        "B": np.array([9.8, 3.2, 0.02]),
        "C": np.array([10.1, 3.2, 0.02]),
    }
    STACK_POS = np.array([10.5, 3.6, 0.0])  # Base de la pila

    def __init__(self, pb_id: int, deploy_pos: np.ndarray, dt: float = 0.05):
        self.id = pb_id
        self.pos = deploy_pos.copy()
        self.arm = PuzzleBotArm()
        self.assigned_box: Optional[str] = None
        self.done = False
        self.dt = dt  # Time step for movement
        
        # Event-based synchronization
        self.waiting_for_event = False
        self.completed_event = None  # Which event this PuzzleBot completed
        
        # State machine for non-blocking operation
        self.state = "IDLE"
        self.target_box_pos = None
        self.stack_target_pos = None 

    def move_to(self, target: np.ndarray, v: float = 0.2, dt: float = 0.05):
        steps = int(np.linalg.norm(target[:2] - self.pos[:2]) / (v * dt)) + 1
        for _ in range(steps):
            delta = target[:2] - self.pos[:2]
            dist = np.linalg.norm(delta)
            if dist < 0.02:
                break
            self.pos[:2] += v * dt * delta / (dist + 1e-9)

    def pick_and_stack_nonblocking(
        self,
        box_name: str,
        stack_height: float,
        exclusion_zones: List[Tuple[np.ndarray, float]],
        event_flags: dict,
        obstacles: List[Tuple[np.ndarray, float]] = None
    ) -> Tuple[bool, float]:
        """Event-based non-blocking version of pick_and_stack."""
        
        # Check for event dependencies
        if box_name == "B" and not event_flags.get("C_completed", False):
            return False, stack_height
        elif box_name == "A" and not event_flags.get("B_completed", False):
            return False, stack_height
            
        box_pos = self.TABLE_BOXES_POS[box_name]

        # Check exclusion zones
        for (center, radius) in exclusion_zones:
            if np.linalg.norm(box_pos[:2] - center[:2]) < radius:
                return False, stack_height

        # State machine for non-blocking operation
        if self.state == "IDLE":
            print(f"\n[PB{self.id}] Iniciando recogida de caja {box_name}")
            self.target_box_pos = box_pos
            self.stack_target_pos = self.STACK_POS + np.array([0, 0, stack_height])
            self.state = "MOVING_TO_BOX"
            
        elif self.state == "MOVING_TO_BOX":
            approach = np.array([box_pos[0] - 0.12, box_pos[1], 0.0])
            if self._step_move_to(approach, obstacles=obstacles):
                self.state = "GRASPING"
                self.arm.reset()
            return False, stack_height
            
        elif self.state == "GRASPING":
            arm_target = np.array([0.08, 0.0, 0.02])
            success = self.arm.grasp_box(arm_target, grip_force=3.0)
            if success:
                self.state = "MOVING_TO_STACK"
            else:
                self.state = "IDLE"
            return False, stack_height
            
        elif self.state == "MOVING_TO_STACK":
            stack_approach = np.array([self.stack_target_pos[0] - 0.12, self.stack_target_pos[1], 0.0])
            if self._step_move_to(stack_approach, obstacles=obstacles):
                self.state = "PLACING"
            return False, stack_height
            
        elif self.state == "PLACING":
            arm_stack_target = np.array([0.08, 0.0, max(stack_height + 0.01, 0.02)])
            placed = self.arm.place_box(arm_stack_target)
            if placed:
                new_height = stack_height + SmallBox.BOX_HEIGHT
                self.done = True
                self.completed_event = f"{box_name}_completed"
                self.state = "DONE"
                print(f"[PB{self.id}] ✓ Caja {box_name} apilada. Evento: {self.completed_event}")
                return True, new_height
            else:
                self.state = "IDLE"
            return False, stack_height
            
        return False, stack_height
    
    def _step_move_to(self, target: np.ndarray, v: float = 0.2,
                       obstacles: List[Tuple[np.ndarray, float]] = None) -> bool:
        """Single step of movement with obstacle avoidance."""
        delta = target[:2] - self.pos[:2]
        dist = np.linalg.norm(delta)
        if dist < 0.02:
            return True
        
        move_dir = delta / (dist + 1e-9)
        
        # Obstacle avoidance: if near an obstacle, add perpendicular steering
        if obstacles:
            for (obs_pos, obs_radius) in obstacles:
                to_obs = obs_pos[:2] - self.pos[:2]
                obs_dist = np.linalg.norm(to_obs)
                if obs_dist < obs_radius:
                    # Perpendicular to obstacle direction (go around)
                    perp = np.array([-to_obs[1], to_obs[0]]) / (obs_dist + 1e-9)
                    # Push away from obstacle
                    repel = -to_obs / (obs_dist + 1e-9)
                    move_dir = move_dir + 1.5 * perp + 0.5 * repel
                    move_dir = move_dir / (np.linalg.norm(move_dir) + 1e-9)
        
        self.pos[:2] += v * self.dt * move_dir
        return False
        
    def pick_and_stack(
        self,
        box_name: str,
        stack_height: float,
        sim_time: float,
        exclusion_zones: List[Tuple[np.ndarray, float]]
    ) -> Tuple[bool, float]:
        """Legacy blocking version for compatibility."""
        
        if sim_time < self.time_slot_start:
            return False, stack_height

        box_pos = self.TABLE_BOXES_POS[box_name]

        for (center, radius) in exclusion_zones:
            if np.linalg.norm(box_pos[:2] - center[:2]) < radius:
                print(f"[PB{self.id}] Zona de exclusión activa — esperando...")
                return False, stack_height

        approach = np.array([box_pos[0] - 0.12, box_pos[1], 0.0])
        print(f"\n[PB{self.id}] Recogiendo caja {box_name} en {box_pos[:2]}")
        self.move_to(approach)
        self.arm.reset()

        arm_target = np.array([0.13, 0.0, 0.02])
        success = self.arm.grasp_box(arm_target, grip_force=3.0)
        if not success:
            return False, stack_height

        stack_pos_3d = self.STACK_POS + np.array([0, 0, stack_height])
        print(f"[PB{self.id}] Apilando caja {box_name} en altura {stack_height:.3f} m")
        stack_approach = np.array([stack_pos_3d[0] - 0.12, stack_pos_3d[1], 0.0])
        self.move_to(stack_approach)

        arm_stack_target = np.array([0.13, 0.0, max(stack_height + 0.01, 0.02)])
        placed = self.arm.place_box(arm_stack_target)
        if placed:
            new_height = stack_height + SmallBox.BOX_HEIGHT
            self.done = True
            return True, new_height

        return False, stack_height


# ---------------------------------------------------------------------------
# Coordinador Principal
# ---------------------------------------------------------------------------

class Coordinator:
    """Máquina de estados que orquesta las tres fases del almacén robótico."""

    WORK_ZONE = np.array([11.0, 3.6])
    TABLE_POS = np.array([10.0, 3.6, 0.75])  # Mesa de trabajo (z=0.75m)

    def __init__(self, dt: float = 0.02):
        self.dt = dt
        self.phase = Phase.IDLE
        self.metrics: dict = {}

        self.husky = HuskyPusher(slip_factor=0.05, dt=dt)
        self.anymal = ANYmalGait(dt=dt)

        # XArms acercados a la zona de llegada del ANYmal para garantizar alcance
        self.xarms = [
            XArm(1, base_pos=np.array([10.8, 3.3, 0.0])),
            XArm(2, base_pos=np.array([10.8, 3.9, 0.0])),
        ]

        pb_positions = [
            np.array([9.0, 3.6, 0.0]),
            np.array([9.0, 4.0, 0.0]),
            np.array([9.0, 3.2, 0.0]),
        ]
        self.puzzlebots = [PuzzleBot(i, pb_positions[i], dt=dt) for i in range(3)]

        self.stack_order = ["C", "B", "A"]
        for pb, box in zip(self.puzzlebots, self.stack_order):
            pb.assigned_box = box

        self.stack_height = 0.0
        self.sim_time = 0.0

    def _transition(self, new_phase: Phase):
        print(f"\n{'='*50}")
        print(f"  TRANSICIÓN: {self.phase.name} → {new_phase.name}")
        print(f"{'='*50}")
        self.phase = new_phase

    def _run_phase1(self) -> bool:
        t0 = self.sim_time
        success = self.husky.clear_corridor()
        self.metrics["phase1_time"] = self.sim_time - t0
        self.metrics["phase1_success"] = success
        return success

    def _run_phase2(self) -> bool:
        t0 = self.sim_time
        success = self.anymal.transport_puzzlebots()
        self.metrics["phase2_time"] = self.sim_time - t0
        self.metrics["phase2_success"] = success
        self.metrics["phase2_final_error"] = float(
            np.linalg.norm(self.anymal.state.pos2d - self.WORK_ZONE)
        )
        return success

    def _run_xarm_unload(self) -> bool:
        print("\n" + "=" * 50)
        print("  EXTRA: XArm descarga PuzzleBots del ANYmal")
        print("=" * 50)

        anymal_pos = self.anymal.state.pos2d
        table_positions = [
            np.array([9.2, 3.6, self.TABLE_POS[2]]),
            np.array([9.4, 3.6, self.TABLE_POS[2]]),
            np.array([9.6, 3.6, self.TABLE_POS[2]]),
        ]

        ok = self.xarms[0].pick_from_anymal(0, anymal_pos)
        ok &= self.xarms[0].place_on_table(table_positions[0])

        ok &= self.xarms[0].pick_from_anymal(1, anymal_pos)
        ok &= self.xarms[0].place_on_table(table_positions[1])

        ok2 = self.xarms[1].pick_from_anymal(2, anymal_pos)
        ok2 &= self.xarms[1].place_on_table(table_positions[2])

        for i, pb in enumerate(self.puzzlebots):
            pb.pos = table_positions[i]

        success = ok and ok2
        self.metrics["xarm_unload_success"] = success
        print(f"[XArm] Descarga: {'✓ EXITOSA' if success else '✗ CON ERRORES'}")
        return success

    def _run_phase3(self) -> bool:
        print("\n" + "=" * 50)
        print("  FASE 3: Apilado cooperativo — PuzzleBots")
        print(f"  Orden: C (abajo) → B (medio) → A (arriba)")
        print("=" * 50)

        t0 = self.sim_time
        stacking_done = [False, False, False]
        max_sim_time = 60.0
        dt_phase3 = 0.05
        
        # Event-based synchronization flags
        event_flags = {
            "C_completed": False,
            "B_completed": False,
            "A_completed": False
        }

        while not all(stacking_done) and (self.sim_time - t0) < max_sim_time:
            for i, pb in enumerate(self.puzzlebots):
                if stacking_done[i]:
                    continue

                exclusion = [
                    (self.puzzlebots[j].pos, 0.25)
                    for j in range(3)
                    if j != i and not stacking_done[j]
                ]

                # Use event-based non-blocking version
                success, new_h = pb.pick_and_stack_nonblocking(
                    pb.assigned_box, self.stack_height,
                    exclusion, event_flags
                )

                if success:
                    self.stack_height = new_h
                    stacking_done[i] = True
                    
                    # Update event flags
                    if pb.completed_event:
                        event_flags[pb.completed_event] = True
                        print(f"[Coordinator] Evento activado: {pb.completed_event}")
                    
                    print(f"[PB{i}] ✓ Caja {pb.assigned_box} apilada. Altura pila = {self.stack_height:.3f} m")

            self.sim_time += dt_phase3

        expected_height = SmallBox.BOX_HEIGHT * 3
        height_ok = abs(self.stack_height - expected_height) < 0.01
        all_stacked = all(stacking_done)

        self.metrics["phase3_time"] = self.sim_time - t0
        self.metrics["phase3_success"] = all_stacked and height_ok
        self.metrics["stack_height_final"] = self.stack_height
        self.metrics["stack_order_ok"] = True
        self.metrics["event_sync_used"] = True

        if all_stacked:
            print(f"\n[Coordinator] ✓ Pila completa: C-B-A | Altura={self.stack_height:.3f} m")
            print(f"[Coordinator] ✓ Sincronización por eventos: C→B→A")
        else:
            failed = [self.stack_order[i] for i, ok in enumerate(stacking_done) if not ok]
            print(f"[Coordinator] ✗ Cajas no apiladas: {failed}")

        return all_stacked and height_ok

    def run(self) -> bool:
        print("\n" + "╔" + "═"*48 + "╗")
        print("║  COORDINADOR — Almacén Robótico Autónomo       ║")
        print("╚" + "═"*48 + "╝")

        t_start = self.sim_time

        self._transition(Phase.PHASE1_HUSKY)
        ok1 = self._run_phase1()
        if not ok1:
            self._transition(Phase.ERROR)
            print("[Coordinator] ✗ Fase 1 falló. Misión abortada.")
            return False
        self.sim_time += self.metrics.get("phase1_time", 0)

        self._transition(Phase.PHASE2_ANYMAL)
        ok2 = self._run_phase2()
        self.sim_time += self.metrics.get("phase2_time", 0)
        if not ok2:
            print("[Coordinator] ⚠ ANYmal no llegó al destino exacto, continuando...")

        self._transition(Phase.XARM_UNLOAD)
        self._run_xarm_unload()

        self._transition(Phase.PHASE3_PUZZLEBOTS)
        ok3 = self._run_phase3()

        self._transition(Phase.DONE)
        self.metrics["total_time"] = self.sim_time - t_start
        self._print_metrics()

        return ok1 and ok3

    def _print_metrics(self):
        print("\n" + "╔" + "═"*48 + "╗")
        print("║          MÉTRICAS FINALES                      ║")
        print("╠" + "═"*48 + "╣")
        rows = [
            ("Tiempo total",        f"{self.metrics.get('total_time',0):.1f} s"),
            ("Fase 1 éxito",        "✓" if self.metrics.get("phase1_success") else "✗"),
            ("Fase 1 tiempo",       f"{self.metrics.get('phase1_time',0):.1f} s"),
            ("Fase 2 éxito",        "✓" if self.metrics.get("phase2_success") else "✗"),
            ("Fase 2 tiempo",       f"{self.metrics.get('phase2_time',0):.1f} s"),
            ("Fase 2 error final",  f"{self.metrics.get('phase2_final_error',0):.4f} m"),
            ("XArm descarga",       "✓" if self.metrics.get("xarm_unload_success") else "✗"),
            ("Fase 3 éxito",        "✓" if self.metrics.get("phase3_success") else "✗"),
            ("Fase 3 tiempo",       f"{self.metrics.get('phase3_time',0):.1f} s"),
            ("Altura pila final",   f"{self.metrics.get('stack_height_final',0):.3f} m"),
            ("Orden C-B-A",         "✓" if self.metrics.get("stack_order_ok") else "✗"),
        ]
        for k, v in rows:
            print(f"║  {k:<28} {v:>15} ║")
        print("╚" + "═"*48 + "╝")

## Simulador 2D - Visualización

Renderiza la simulación completa con matplotlib:
- Animación en tiempo real de las 3 fases
- Panel de métricas en vivo
- Generación de gráficas finales


In [19]:
"""
sim.py — Simulador 2D (matplotlib) del escenario completo con visualización animada.

Muestra las tres fases en tiempo real:
    1. Husky empujando cajas
    2. ANYmal caminando con PuzzleBots
    3. PuzzleBots apilando cajas A-B-C
"""

import numpy as np
import os
import matplotlib
try:
    matplotlib.use("TkAgg")
except Exception:
    pass
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, Rectangle, FancyBboxPatch
from typing import List



# ---------------------------------------------------------------------------
# Paleta de colores
# ---------------------------------------------------------------------------
COLORS = {
    "bg":          "#0f1117",
    "corridor":    "#1a2340",
    "work_zone":   "#1a3320",
    "start_zone":  "#2a1a30",
    "husky":       "#e74c3c",
    "anymal":      "#3498db",
    "puzzlebot":   "#2ecc71",
    "box_large":   "#e67e22",
    "box_a":       "#9b59b6",
    "box_b":       "#1abc9c",
    "box_c":       "#f39c12",
    "stack":       "#ecf0f1",
    "grid":        "#2c3e50",
    "text":        "#ecf0f1",
    "xarm":        "#e91e63",
    "success":     "#27ae60",
    "warning":     "#f39c12",
}


class Sim2D:
    """Simulador 2D animado del almacén robótico."""

    # Dimensiones del escenario
    X_TOTAL = 13.0
    Y_MIN, Y_MAX = -3.0, 6.0

    def __init__(self, dt: float = 0.05, save_gif: bool = False):
        self.dt = dt
        self.save_gif = save_gif
        self.frames: List = []

        # Sistemas
        self.husky = HuskyPusher(slip_factor=0.05, dt=dt)
        self.anymal = ANYmalGait(dt=dt)

        # Instanciar coordinador para la lógica de la fase 3
        self.coord = Coordinator(dt=dt)

        # Estado visual
        self.current_phase = "INICIO"
        self.phase_log: List[str] = []
        self.stack_boxes: List[dict] = []  # Cajas en la pila

        # Figura
        self.fig, self.axes = plt.subplots(1, 2, figsize=(16, 8),
                                            gridspec_kw={"width_ratios": [3, 1]})
        self.fig.patch.set_facecolor(COLORS["bg"])
        self.ax = self.axes[0]
        self.ax_info = self.axes[1]
        self._setup_scene()

    # ------------------------------------------------------------------
    # Configuración de la escena
    # ------------------------------------------------------------------

    def _setup_scene(self):
        """Configura el fondo fijo del escenario."""
        ax = self.ax
        ax.set_facecolor(COLORS["bg"])
        ax.set_xlim(-1, self.X_TOTAL)
        ax.set_ylim(self.Y_MIN, self.Y_MAX)
        ax.set_aspect("equal")
        ax.set_title("Almacén Robótico Autónomo — TE3002B",
                     color=COLORS["text"], fontsize=13, fontweight="bold", pad=12)

        # Grid
        ax.grid(True, color=COLORS["grid"], linewidth=0.4, alpha=0.5)
        ax.tick_params(colors=COLORS["text"])
        for spine in ax.spines.values():
            spine.set_edgecolor(COLORS["grid"])

        # Zona de inicio
        start = Rectangle((-0.5, -1.5), 2.0, 3.0, linewidth=2,
                           edgecolor=COLORS["start_zone"], facecolor=COLORS["start_zone"],
                           alpha=0.4, label="Zona de Inicio")
        ax.add_patch(start)
        ax.text(0.5, 0, "INICIO", color=COLORS["text"], fontsize=9,
                ha="center", va="center", alpha=0.7)

        # Corredor (6×2 m)
        corridor = Rectangle((1.0, -1.0), 6.0, 2.0, linewidth=2,
                              edgecolor="#4a90d9", facecolor=COLORS["corridor"],
                              alpha=0.5, label="Corredor")
        ax.add_patch(corridor)
        ax.text(4.0, 0, "CORREDOR", color=COLORS["text"], fontsize=9,
                ha="center", va="center", alpha=0.7)

        # Zona de trabajo
        work = Rectangle((7.5, 1.0), 5.0, 5.0, linewidth=2,
                          edgecolor="#4aad72", facecolor=COLORS["work_zone"],
                          alpha=0.4, label="Zona de Trabajo")
        ax.add_patch(work)
        ax.text(10.0, 5.2, "ZONA DE TRABAJO", color=COLORS["text"], fontsize=9,
                ha="center", va="center", alpha=0.7)

        # Destino del ANYmal
        ax.plot(*[11.5, 4.8], "o", color="#4a90d9", markersize=10, alpha=0.5)
        ax.text(11.5, 4.4, "Destino\nANYmal", color="#4a90d9", fontsize=7,
                ha="center", alpha=0.8)

        # Pila destino
        ax.plot(*[10.5, 3.6], "^", color=COLORS["stack"], markersize=10, alpha=0.6)
        ax.text(10.5, 3.1, "Pila\nDestino", color=COLORS["stack"], fontsize=7,
                ha="center", alpha=0.8)

        # Leyenda
        legend_elements = [
            mpatches.Patch(color=COLORS["husky"],     label="Husky A200"),
            mpatches.Patch(color=COLORS["anymal"],    label="ANYmal"),
            mpatches.Patch(color=COLORS["puzzlebot"], label="PuzzleBot"),
            mpatches.Patch(color=COLORS["box_large"], label="Caja grande"),
            mpatches.Patch(color=COLORS["xarm"],      label="XArm"),
        ]
        ax.legend(handles=legend_elements, loc="lower right",
                  facecolor=COLORS["bg"], edgecolor=COLORS["grid"],
                  labelcolor=COLORS["text"], fontsize=8)

        # Panel info
        self.ax_info.set_facecolor(COLORS["bg"])
        self.ax_info.set_xlim(0, 1)
        self.ax_info.set_ylim(0, 1)
        self.ax_info.axis("off")

    # ------------------------------------------------------------------
    # Dibujo de robots y objetos
    # ------------------------------------------------------------------

    def _draw_robot(self, ax, x, y, theta, color, size=0.3, label=""):
        """Dibuja un robot como un rectángulo con flecha de dirección."""
        # Cuerpo
        rect = FancyBboxPatch(
            (x - size / 2, y - size / 2), size, size,
            boxstyle="round,pad=0.02",
            facecolor=color, edgecolor="white", linewidth=1, alpha=0.9
        )
        ax.add_patch(rect)
        # Flecha de dirección
        dx = size * 0.6 * np.cos(theta)
        dy = size * 0.6 * np.sin(theta)
        ax.annotate("", xy=(x + dx, y + dy), xytext=(x, y),
                    arrowprops=dict(arrowstyle="->", color="white", lw=1.5))
        if label:
            ax.text(x, y - size * 0.7, label, color=COLORS["text"],
                    fontsize=7, ha="center")

    def _draw_box_large(self, ax, box: Box):
        """Dibuja una caja grande."""
        color = COLORS["box_large"] if not box.cleared else "#555555"
        rect = Rectangle(
            (box.x - box.width / 2, box.y - box.height / 2),
            box.width, box.height,
            facecolor=color, edgecolor="white", linewidth=0.8, alpha=0.85
        )
        ax.add_patch(rect)
        ax.text(box.x, box.y, box.id, color="white", fontsize=8,
                ha="center", va="center", fontweight="bold")

    def _draw_anymal_with_pbs(self, ax, anymal_state, pb_positions):
        """Dibuja el ANYmal con los PuzzleBots en el dorso."""
        x, y, theta = anymal_state.x, anymal_state.y, anymal_state.theta
        # Cuerpo ANYmal
        body_len, body_w = 0.55, 0.34
        rect = FancyBboxPatch(
            (x - body_len / 2, y - body_w / 2), body_len, body_w,
            boxstyle="round,pad=0.03",
            facecolor=COLORS["anymal"], edgecolor="white", linewidth=1.5, alpha=0.9
        )
        ax.add_patch(rect)
        ax.text(x, y + body_w / 2 + 0.05, "ANYmal", color=COLORS["text"],
                fontsize=7, ha="center")

        # Patas (simplificadas)
        for leg_name, leg_state in anymal_state.legs.items():
            lx, ly = leg_state.foot_pos_world[0], leg_state.foot_pos_world[1]
            c = COLORS["success"] if leg_state.det_J > 1e-3 else COLORS["warning"]
            ax.plot(lx, ly, "o", color=c, markersize=4, alpha=0.8)
            ax.plot([x, lx], [y, ly], "-", color=COLORS["anymal"], linewidth=0.8, alpha=0.4)

        # PuzzleBots en el dorso
        for i, pb_pos in enumerate(pb_positions):
            ax.plot(pb_pos[0], pb_pos[1], "s", color=COLORS["puzzlebot"],
                    markersize=7, alpha=0.9)

    def _draw_info_panel(self, phase_name: str, metrics: dict):
        """Actualiza el panel lateral con métricas."""
        ax = self.ax_info
        ax.clear()
        ax.set_facecolor(COLORS["bg"])
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis("off")

        y_pos = 0.95
        ax.text(0.5, y_pos, "ESTADO", color=COLORS["text"],
                fontsize=11, ha="center", fontweight="bold")
        y_pos -= 0.07

        ax.text(0.5, y_pos, phase_name, color=COLORS["warning"],
                fontsize=10, ha="center", fontweight="bold")
        y_pos -= 0.08

        ax.axhline(y=y_pos + 0.01, color=COLORS["grid"], linewidth=0.8, xmin=0.05, xmax=0.95)
        y_pos -= 0.04

        for key, val in metrics.items():
            ax.text(0.05, y_pos, f"{key}:", color=COLORS["text"], fontsize=8)
            color = COLORS["success"] if "✓" in str(val) else (
                    COLORS["warning"] if "✗" in str(val) else COLORS["text"])
            ax.text(0.95, y_pos, str(val), color=color, fontsize=8, ha="right")
            y_pos -= 0.055
            if y_pos < 0.05:
                break

    def _refresh_live_view(
        self,
        phase_name: str,
        metrics: dict,
        show_anymal: bool = False,
        pb_positions: List[np.ndarray] | None = None,
    ):
        """Redibuja la escena completa en la ventana interactiva."""
        self.ax.clear()
        self.ax_info.clear()
        self._setup_scene()

        for box in self.husky.boxes:
            self._draw_box_large(self.ax, box)

        self._draw_robot(
            self.ax,
            self.husky.state.x,
            self.husky.state.y,
            self.husky.state.theta,
            COLORS["husky"],
            size=0.35,
            label="Husky",
        )

        if show_anymal:
            self._draw_anymal_with_pbs(self.ax, self.anymal.state, [])
        
        if pb_positions is not None:
            for i, pb_pos in enumerate(pb_positions):
                self.ax.plot(pb_pos[0], pb_pos[1], "s", color=COLORS["puzzlebot"],
                             markersize=10, zorder=6)
                self.ax.text(pb_pos[0], pb_pos[1] - 0.35, f"PB{i}",
                             color=COLORS["puzzlebot"], fontsize=7, ha="center")

        for box_info in self.stack_boxes:
            bname = box_info["name"]
            bpos = box_info["pos"]
            self.ax.add_patch(Rectangle(
                (bpos[0] - 0.05, bpos[1] - 0.05), 0.10, 0.10,
                facecolor={"A": COLORS["box_a"], "B": COLORS["box_b"], "C": COLORS["box_c"]}.get(bname, "white"),
                edgecolor="white", linewidth=0.8, alpha=0.95, zorder=7
            ))
            self.ax.text(bpos[0], bpos[1], bname, color="white", fontsize=7,
                         ha="center", va="center", fontweight="bold")

        self._draw_info_panel(phase_name, metrics)
        self.fig.canvas.draw_idle()
        self.fig.canvas.flush_events()
        plt.pause(0.001)

    # ------------------------------------------------------------------
    # Simulación y animación
    # ------------------------------------------------------------------

    def run_and_save(self, output_path: str = "sim_output.png", live_display: bool = True):
        """Ejecuta la simulación completa, muestra la vista en vivo y guarda frames clave."""
        print("\n[Sim2D] Iniciando simulación completa...")

        if live_display:
            plt.ion()
            plt.show(block=False)

        all_frames_data = []

        # ── Fase 1: Husky (Non-blocking) ───────────────────────────────────
        print("[Sim2D] Fase 1: Husky despejando corredor (animación fluida)...")
        self.current_phase = "FASE 1: Husky despeja corredor"
        print("[ML] Husky - Logistic Regression activado")
        
        # Reset Husky state machine for non-blocking operation
        self.husky.nav_state = "IDLE"
        self.husky.target_box = None
        self.husky.target_pos = None
        self.husky.pushing = False
        self.husky.returning = False
        
        phase1_complete = False
        for step in range(1200):  # More steps for smoother animation
            # Use non-blocking corridor clearing
            if not phase1_complete:
                phase1_complete = self.husky.clear_corridor_step()

            # Capturar frame cada 15 pasos (more frequent for smoother animation)
            if step % 15 == 0:
                frame_data = {
                    "phase": "phase1",
                    "husky": (self.husky.state.x, self.husky.state.y, self.husky.state.theta),
                    "boxes": [(b.x, b.y, b.cleared) for b in self.husky.boxes],
                    "anymal": None,
                    "pbs": [],
                }
                all_frames_data.append(frame_data)

            if live_display and (step % 3 == 0 or step == 1199):  # Update display more frequently
                cleared_count = sum(b.cleared for b in self.husky.boxes)
                nav_state = self.husky.nav_state
                metrics = {
                    "Cajas despejadas": f"{cleared_count}/{len(self.husky.boxes)}",
                    "Tiempo Husky": f"{self.husky.time:.1f} s",
                    "Estado navegación": nav_state,
                }
                self._refresh_live_view(self.current_phase, metrics, show_anymal=False)

            if phase1_complete:
                print("[Sim2D] ✓ Corredor despejado (animación fluida)")
                break

        if live_display:
            cleared_count = sum(b.cleared for b in self.husky.boxes)
            self._refresh_live_view(
                self.current_phase,
                {
                    "Cajas despejadas": f"{cleared_count}/{len(self.husky.boxes)}",
                    "Tiempo Husky": f"{self.husky.time:.1f} s",
                },
                show_anymal=False,
            )

        # ── Fase 2: ANYmal ────────────────────────────────────────────
        print("[Sim2D] Fase 2: ANYmal caminando...")
        self.current_phase = "FASE 2: ANYmal transportando PuzzleBots"
        
        # ANYmal empieza detrás del Husky, lo rodea por abajo
        self.anymal.state.x = -0.5
        self.anymal.state.y = -1.5
        
        # Waypoints: rodear Husky → corredor → zona de trabajo
        waypoints = [
            np.array([1.5, -1.5]),   # Rodear al Husky por abajo
            np.array([2.0, 0.0]),    # Entrar al corredor
            np.array([7.0, 0.0]),    # Final del corredor
            np.array([8.5, 2.0]),    # Giro hacia zona de trabajo
            np.array([11.5, 4.8]),   # Destino final (lejos de la pila)
        ]
        
        total_dist = sum(
            np.linalg.norm(waypoints[i] - (waypoints[i-1] if i > 0 else self.anymal.state.pos2d))
            for i in range(len(waypoints))
        )
        eta = ml_system.anymal_predict_eta(total_dist, 6.0)
        print(f"[ML] ANYmal ETA: {eta:.1f}s para {total_dist:.2f}m (Linear Regression)")

        for dest in waypoints:
            for step in range(2000):
                dx = dest[0] - self.anymal.state.x
                dy = dest[1] - self.anymal.state.y
                dist = np.hypot(dx, dy)
                if dist < 0.3:
                    break
                direction = np.array([dx, dy]) / (dist + 1e-9)
                v = min(0.4, dist * 2.0)
                self.anymal._step(v, direction)

                if step % 50 == 0:
                    pb_offsets = [np.array([self.anymal.state.x + (i-1)*0.15, self.anymal.state.y, 0.5])
                                  for i in range(3)]
                    frame_data = {
                        "phase": "phase2",
                        "husky": (self.husky.state.x, self.husky.state.y, self.husky.state.theta),
                        "boxes": [(b.x, b.y, b.cleared) for b in self.husky.boxes],
                        "anymal": self.anymal.state,
                        "pbs": pb_offsets,
                        "det_J": self.anymal.det_J_summary(),
                    }
                    all_frames_data.append(frame_data)

                if live_display and (step % 10 == 0):
                    pb_offsets = [np.array([self.anymal.state.x + (i - 1) * 0.15, self.anymal.state.y, 0.5])
                                  for i in range(3)]
                    metrics = {
                        "Distancia al destino": f"{np.linalg.norm(waypoints[-1] - self.anymal.state.pos2d):.2f} m",
                        "Posición": f"({self.anymal.state.x:.2f}, {self.anymal.state.y:.2f})",
                    }
                    self._refresh_live_view(self.current_phase, metrics, show_anymal=True, pb_positions=pb_offsets)
        
        print(f"[Sim2D] ✓ ANYmal llegó. Error={np.linalg.norm(waypoints[-1] - self.anymal.state.pos2d):.4f} m")

        # ── Fase 3: PuzzleBots (Real stacking with force control) ────────
        print("[Sim2D] Fase 3: PuzzleBots apilando con control de fuerza real...")
        self.current_phase = "FASE 3: PUZZLEBOTS"
        
        # PuzzleBots bajan del ANYmal uno por uno (animado)
        anymal_x, anymal_y = self.anymal.state.x, self.anymal.state.y
        # Posiciones de trabajo cerca de las cajas (no del ANYmal)
        pb_targets = [
            np.array([9.3, 3.0, 0.0]),   # Cerca de caja C
            np.array([9.6, 3.0, 0.0]),   # Cerca de caja B
            np.array([9.9, 3.0, 0.0]),   # Cerca de caja A
        ]
        stack_order = ["C", "B", "A"]
        
        # Todos empiezan en la posición del ANYmal
        pb_positions_phase3 = [
            np.array([anymal_x, anymal_y, 0.0]) for _ in range(3)
        ]
        for i, pb in enumerate(self.coord.puzzlebots):
            pb.state = "IDLE"
            pb.done = False
            pb.completed_event = None
            pb.pos = np.array([anymal_x, anymal_y, 0.0])
        
        # Animar descenso uno por uno hacia zona de trabajo
        for i, pb in enumerate(self.coord.puzzlebots):
            target = pb_targets[i]
            print(f"[Sim2D] PB{i} bajando del ANYmal hacia zona de trabajo...")
            for s in range(800):
                delta = target[:2] - pb.pos[:2]
                dist = np.linalg.norm(delta)
                if dist < 0.05:
                    break
                move_dir = delta / (dist + 1e-9)
                
                # Evitar al ANYmal
                to_anymal = np.array([anymal_x, anymal_y]) - pb.pos[:2]
                d_anymal = np.linalg.norm(to_anymal)
                if d_anymal < 0.6:
                    perp = np.array([-to_anymal[1], to_anymal[0]]) / (d_anymal + 1e-9)
                    repel = -to_anymal / (d_anymal + 1e-9)
                    move_dir = move_dir + 1.5 * perp + 0.5 * repel
                    move_dir = move_dir / (np.linalg.norm(move_dir) + 1e-9)
                
                # Evitar otros PBs
                for j, other_pb in enumerate(self.coord.puzzlebots):
                    if j == i:
                        continue
                    to_other = other_pb.pos[:2] - pb.pos[:2]
                    d_other = np.linalg.norm(to_other)
                    if d_other < 0.35:
                        repel = -to_other / (d_other + 1e-9)
                        move_dir = move_dir + 0.8 * repel
                        move_dir = move_dir / (np.linalg.norm(move_dir) + 1e-9)
                
                pb.pos[:2] += 0.2 * self.dt * move_dir
                pb_positions_phase3[i] = pb.pos.copy()
                if live_display and s % 8 == 0:
                    self._refresh_live_view(
                        self.current_phase,
                        {"Estado": f"PB{i} bajando del ANYmal..."},
                        show_anymal=True, pb_positions=pb_positions_phase3,
                    )
            pb.pos = target.copy()
            pb_positions_phase3[i] = pb.pos.copy()
            zone = ml_system.puzzlebot_get_zone(pb.pos)
            print(f"[ML] PuzzleBot {pb.id} en posición → zona: {zone} (K-Means)")
        
        # Event-based synchronization
        event_flags = {
            "C_completed": False,
            "B_completed": False,
            "A_completed": False
        }
        
        stack_height = 0.0
        phase3_complete = False
        
        for step in range(2400):
            if not phase3_complete:
                for i, pb in enumerate(self.coord.puzzlebots):
                    if pb.done:
                        continue
                    
                    # Solo ANYmal como obstáculo (event sync ya previene conflictos entre PBs)
                    obstacles = [(np.array([anymal_x, anymal_y]), 0.6)]
                    success, new_h = pb.pick_and_stack_nonblocking(
                        pb.assigned_box, stack_height, [], event_flags,
                        obstacles=obstacles
                    )
                    
                    if success:
                        stack_height = new_h
                        
                        # Update event flags
                        if pb.completed_event:
                            event_flags[pb.completed_event] = True
                        
                        # Add to stack visualization
                        stack_pos = np.array([10.5, 3.6, stack_height - SmallBox.BOX_HEIGHT])
                        self.stack_boxes.append({"name": pb.assigned_box, "pos": stack_pos.copy()})
                        
                        # PuzzleBot se aparta un poco de la pila (sin teleport)
                        offset = np.array([-0.3, 0.3 * (i - 1), 0.0])
                        pb.pos = pb.pos + offset
                        pb_positions_phase3[i] = pb.pos.copy()
                        print(f"[Sim2D] ✓ Caja {pb.assigned_box} apilada — PB{i} se aparta")

                
                # Check if phase 3 is complete
                phase3_complete = all(pb.done for pb in self.coord.puzzlebots)
            
            # Update positions for visualization
            for i, pb in enumerate(self.coord.puzzlebots):
                pb_positions_phase3[i] = pb.pos.copy()
            
            # Capture frames more frequently
            if step % 20 == 0:
                frame_data = {
                    "phase": "phase3",
                    "husky": (self.husky.state.x, self.husky.state.y, 0),
                    "boxes": [(b.x, b.y, b.cleared) for b in self.husky.boxes],
                    "anymal": self.anymal.state,
                    "pbs": pb_positions_phase3.copy(),
                    "stack": list(self.stack_boxes),
                    "stack_height": stack_height,
                    "events": event_flags.copy(),
                }
                all_frames_data.append(frame_data)
            
            if live_display and (step % 5 == 0 or step == 2399):
                active_pb = sum(1 for pb in self.coord.puzzlebots if not pb.done)
                metrics = {
                    "Altura de pila": f"{stack_height:.3f} m",
                    "PuzzleBots activos": f"{active_pb}/3",
                    "Eventos completados": f"{sum(event_flags.values())}/3",
                    "Control de fuerza": "τ=J^T*f",
                }
                self._refresh_live_view(
                    self.current_phase, metrics,
                    show_anymal=True,
                    pb_positions=pb_positions_phase3,
                )
            
            if phase3_complete:
                print("[Sim2D] ✓ Fase 3 completada con control de fuerza real")
                break

        # ── Renderizar frames clave ───────────────────────────────────
        total_time = self.husky.time + 30.0 + 40.0
        predicted = ml_system.coordinator_predict_mission(self.husky.time, 30.0, 40.0)
        print(f"\n[ML] Coordinator - Ridge Regression:")
        print(f"  Tiempo predicho: {predicted:.1f}s")
        
        self._render_composite(all_frames_data, output_path)
        print(f"[Sim2D] Imagen guardada en {output_path}")

        if live_display:
            # Mantener la ventana 3 segundos al finalizar y cerrar automáticamente.
            plt.ioff()
            plt.pause(3.0)
            plt.close(self.fig)

    def _render_composite(self, frames_data: List, output_path: str):
        """Renderiza una imagen compuesta con los momentos clave de la simulación."""
        n_key = min(6, len(frames_data))
        key_indices = np.linspace(0, len(frames_data) - 1, n_key, dtype=int)

        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.patch.set_facecolor(COLORS["bg"])
        fig.suptitle(
            "Almacén Robótico Autónomo — TE3002B  |  Simulación 2D",
            color=COLORS["text"], fontsize=14, fontweight="bold", y=0.98
        )

        phase_labels = {
            "phase1": "Fase 1: Husky despeja corredor",
            "phase2": "Fase 2: ANYmal transporta PuzzleBots",
            "phase3": "Fase 3: PuzzleBots apilan cajas",
        }

        for idx, (ax, frame_idx) in enumerate(zip(axes.flat, key_indices)):
            frame = frames_data[frame_idx]
            self._render_frame(ax, frame, idx + 1, len(key_indices))

        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(output_path, dpi=150, bbox_inches="tight",
                    facecolor=COLORS["bg"], edgecolor="none")
        plt.close(fig)

    def _render_frame(self, ax, frame: dict, frame_n: int, total: int):
        """Dibuja un frame individual en un eje dado."""
        phase = frame["phase"]
        ax.set_facecolor(COLORS["bg"])
        ax.set_xlim(-1, self.X_TOTAL)
        ax.set_ylim(self.Y_MIN, self.Y_MAX)
        ax.set_aspect("equal")
        ax.tick_params(colors=COLORS["text"])
        ax.grid(True, color=COLORS["grid"], linewidth=0.3, alpha=0.4)
        for spine in ax.spines.values():
            spine.set_edgecolor(COLORS["grid"])

        phase_labels = {
            "phase1": "Fase 1: Husky",
            "phase2": "Fase 2: ANYmal",
            "phase3": "Fase 3: PuzzleBots",
        }
        ax.set_title(f"{phase_labels.get(phase, phase)}  [{frame_n}/{total}]",
                     color=COLORS["text"], fontsize=9)

        # Zonas
        ax.add_patch(Rectangle((-0.5, -1.5), 2.0, 3.0, alpha=0.3,
                                facecolor=COLORS["start_zone"], edgecolor="none"))
        ax.add_patch(Rectangle((1.0, -1.0), 6.0, 2.0, alpha=0.3,
                                facecolor=COLORS["corridor"], edgecolor="#4a90d9", linewidth=0.8))
        ax.add_patch(Rectangle((7.5, 1.0), 5.0, 5.0, alpha=0.3,
                                facecolor=COLORS["work_zone"], edgecolor="#4aad72", linewidth=0.8))

        # Cajas grandes
        for (bx, by, cleared) in frame["boxes"]:
            color = "#555555" if cleared else COLORS["box_large"]
            ax.add_patch(Rectangle((bx - 0.2, by - 0.2), 0.4, 0.4,
                                    facecolor=color, edgecolor="white", linewidth=0.8, alpha=0.85))

        # Husky
        hx, hy, ht = frame["husky"]
        ax.plot(hx, hy, "s", color=COLORS["husky"], markersize=10, zorder=5)
        ax.annotate("", xy=(hx + 0.3 * np.cos(ht), hy + 0.3 * np.sin(ht)),
                    xytext=(hx, hy),
                    arrowprops=dict(arrowstyle="->", color="white", lw=1.2))

        # ANYmal
        if frame.get("anymal") is not None:
            anymal = frame["anymal"]
            ax.add_patch(FancyBboxPatch(
                (anymal.x - 0.27, anymal.y - 0.17), 0.55, 0.34,
                boxstyle="round,pad=0.03",
                facecolor=COLORS["anymal"], edgecolor="white", linewidth=1.2, alpha=0.9, zorder=4
            ))

        # PuzzleBots
        for pb_pos in frame.get("pbs", []):
            ax.plot(pb_pos[0], pb_pos[1], "D", color=COLORS["puzzlebot"],
                    markersize=7, zorder=6)

        # Pila de cajas
        box_colors = {"A": COLORS["box_a"], "B": COLORS["box_b"], "C": COLORS["box_c"]}
        for box_info in frame.get("stack", []):
            bname = box_info["name"]
            bpos = box_info["pos"]
            ax.add_patch(Rectangle(
                (bpos[0] - 0.05, bpos[1] - 0.05), 0.10, 0.10,
                facecolor=box_colors.get(bname, "white"),
                edgecolor="white", linewidth=0.8, alpha=0.95, zorder=7
            ))
            ax.text(bpos[0], bpos[1], bname, color="white", fontsize=7,
                    ha="center", va="center", fontweight="bold")

        # Destino ANYmal
        ax.plot(11.5, 4.8, "o", color="#4a90d9", markersize=6, alpha=0.5, zorder=2)
        ax.plot(10.5, 3.6, "^", color=COLORS["stack"], markersize=6, alpha=0.5, zorder=2)


# ---------------------------------------------------------------------------
# Función de visualización rápida de métricas (para el reporte)
# ---------------------------------------------------------------------------

def plot_metrics(anymal: ANYmalGait, husky: HuskyPusher, output_path: str = "metrics.png"):
    """Genera gráficas de métricas para el reporte técnico."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.patch.set_facecolor(COLORS["bg"])
    fig.suptitle("Métricas del Sistema — TE3002B", color=COLORS["text"],
                 fontsize=13, fontweight="bold")

    ax_colors = [COLORS["text"]] * 4
    for i, ax in enumerate(axes.flat):
        ax.set_facecolor("#1a1e2e")
        ax.tick_params(colors=COLORS["text"])
        ax.grid(True, color=COLORS["grid"], linewidth=0.4, alpha=0.5)
        for spine in ax.spines.values():
            spine.set_edgecolor(COLORS["grid"])

    # 1. Trayectoria del ANYmal
    ax = axes[0, 0]
    if anymal.pos_log:
        traj = np.array(anymal.pos_log)
        ax.plot(traj[:, 0], traj[:, 1], color=COLORS["anymal"], linewidth=1.5, label="ANYmal")
        ax.plot(traj[0, 0], traj[0, 1], "go", markersize=8, label="Inicio")
        ax.plot(11.5, 4.8, "r*", markersize=12, label="Destino")
    ax.set_title("Trayectoria ANYmal", color=COLORS["text"])
    ax.set_xlabel("x [m]", color=COLORS["text"])
    ax.set_ylabel("y [m]", color=COLORS["text"])
    ax.legend(facecolor="#1a1e2e", labelcolor=COLORS["text"], fontsize=8)

    # 2. det(J) por pata
    ax = axes[0, 1]
    if anymal.det_J_log:
        t_axis = np.arange(len(anymal.det_J_log)) * anymal.dt
        pata_colors = {"LF": "#e74c3c", "RF": "#3498db", "LH": "#2ecc71", "RH": "#f39c12"}
        for leg, color in pata_colors.items():
            vals = [d.get(leg, 0) for d in anymal.det_J_log]
            ax.plot(t_axis, vals, color=color, linewidth=1.0, label=leg, alpha=0.8)
        ax.axhline(y=1e-3, color="red", linestyle="--", linewidth=1.2, label="|det(J)|_min")
    ax.set_title("det(J) por pata", color=COLORS["text"])
    ax.set_xlabel("Tiempo [s]", color=COLORS["text"])
    ax.set_ylabel("|det(J)|", color=COLORS["text"])
    ax.legend(facecolor="#1a1e2e", labelcolor=COLORS["text"], fontsize=8)
    ax.set_yscale("log")

    # 3. Velocidades Husky
    ax = axes[1, 0]
    if husky.controller.log_v_cmd:
        t_h = np.arange(len(husky.controller.log_v_cmd)) * husky.dt
        ax.plot(t_h, husky.controller.log_v_cmd, color=COLORS["husky"],
                linewidth=1.2, label="v_cmd", alpha=0.9)
        ax.plot(t_h, husky.controller.log_v_meas, color="#ff8888",
                linewidth=1.0, label="v_meas", linestyle="--", alpha=0.8)
    ax.set_title("Husky — Velocidad lineal", color=COLORS["text"])
    ax.set_xlabel("Tiempo [s]", color=COLORS["text"])
    ax.set_ylabel("v [m/s]", color=COLORS["text"])
    ax.legend(facecolor="#1a1e2e", labelcolor=COLORS["text"], fontsize=8)

    # 4. ω Husky
    ax = axes[1, 1]
    if husky.controller.log_w_cmd:
        t_h = np.arange(len(husky.controller.log_w_cmd)) * husky.dt
        ax.plot(t_h, husky.controller.log_w_cmd, color=COLORS["anymal"],
                linewidth=1.2, label="ω_cmd", alpha=0.9)
        ax.plot(t_h, husky.controller.log_w_meas, color="#88ccff",
                linewidth=1.0, label="ω_meas", linestyle="--", alpha=0.8)
    ax.set_title("Husky — Velocidad angular", color=COLORS["text"])
    ax.set_xlabel("Tiempo [s]", color=COLORS["text"])
    ax.set_ylabel("ω [rad/s]", color=COLORS["text"])
    ax.legend(facecolor="#1a1e2e", labelcolor=COLORS["text"], fontsize=8)

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight",
                facecolor=COLORS["bg"], edgecolor="none")
    plt.close(fig)
    print(f"[Sim2D] Métricas guardadas en {output_path}")

## Ejecutar Simulación

Corre la simulación completa. Se generarán archivos en `results/`:
- `sim_output.png`: Visualización de las 3 fases
- `metrics.png`: Gráficas de trayectorias y torques
- `torque_report.json`: Log detallado de control de fuerza


In [20]:
np.random.seed(42)
os.makedirs("results", exist_ok=True)

sim = Sim2D(dt=0.05)
sim.run_and_save("results/sim_output.png", live_display=True)

plot_metrics(sim.anymal, sim.husky, "results/metrics.png")
torque_logger.print_summary()
torque_logger.generate_torque_report("results/torque_report.json")

print("\nSimulación completada")


[Sim2D] Iniciando simulación completa...
[Sim2D] Fase 1: Husky despejando corredor (animación fluida)...
[ML] Husky - Logistic Regression activado

[HuskyPusher] Iniciando aproximación a caja B1
[HuskyPusher] En punto seguro. Alineándose con B1
[HuskyPusher] Posicionado. Empujando...
[HuskyPusher] ✓ Caja B1 despejada

[HuskyPusher] Iniciando aproximación a caja B2
[HuskyPusher] En punto seguro. Alineándose con B2
[HuskyPusher] Posicionado. Empujando...
[HuskyPusher] ✓ Caja B2 despejada

[HuskyPusher] Iniciando aproximación a caja B3
[HuskyPusher] En punto seguro. Alineándose con B3
[HuskyPusher] Posicionado. Empujando...
[HuskyPusher] ✓ Caja B3 despejada

[HuskyPusher] Corredor ✓ DESPEJADO
[Sim2D] ✓ Corredor despejado (animación fluida)
[Sim2D] Fase 2: ANYmal caminando...
[ML] ANYmal ETA: 53.6s para 15.18m (Linear Regression)
[Sim2D] ✓ ANYmal llegó. Error=0.2905 m
[Sim2D] Fase 3: PuzzleBots apilando con control de fuerza real...
[Sim2D] PB0 bajando del ANYmal hacia zona de trabajo...
